# 05 — Qwen2.5-7B Evidence-Validated Valve Extraction — Final

Final extraction architecture:

`clinical note → deterministic valve-context selector → Qwen2.5-7B candidate extraction → deterministic evidence validation → canonical valve record`

Qwen proposes clinically relevant facts. Python then enforces:
- direct evidence IDs,
- procedure completion evidence,
- years only from the cited line,
- model/size support,
- event-specific evidence rules,
- redo / valve-in-valve logic,
- event deduplication.

The model is never trusted to create unsupported chronology or evidence.


## 1. Clean Colab Hugging Face environment — run once, then restart runtime


In [1]:
# IMPORTANT:
# Run this cell ONCE, then use Runtime -> Restart session/runtime.
# After restart, continue from the next cell. Do not rerun this install cell.

%pip uninstall -y transformers tokenizers huggingface_hub

%pip install -q --no-cache-dir \
    "transformers==4.48.3" \
    "tokenizers==0.21.0" \
    "huggingface_hub==0.28.1" \
    "accelerate>=1.2,<2" \
    "bitsandbytes>=0.45,<0.47" \
    json-repair \
    jsonschema \
    pandas \
    openpyxl

print(
    "\nINSTALL COMPLETE. NOW RESTART THE COLAB RUNTIME ONCE, "
    "then continue from the next cell."
)


Found existing installation: transformers 5.16.1
Uninstalling transformers-5.16.1:
  Successfully uninstalled transformers-5.16.1
Found existing installation: tokenizers 0.23.1
Uninstalling tokenizers-0.23.1:
  Successfully uninstalled tokenizers-0.23.1
Found existing installation: huggingface_hub 1.29.0
Uninstalling huggingface_hub-1.29.0:
  Successfully uninstalled huggingface_hub-1.29.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 597.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 197.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 464.1/464.1 kB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 181.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

## 2. Imports and environment sanity check — run after restart


In [2]:
import os
os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True",
)

from pathlib import Path
import gc
import json
import re
import traceback
import warnings
import sys

import numpy as np
import pandas as pd
import torch
import transformers
import tokenizers
import huggingface_hub

from IPython.display import display
from json_repair import repair_json

warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())

assert transformers.__version__ == "4.48.3", (
    "Wrong Transformers version. Restart the runtime after the install cell."
)
assert tokenizers.__version__ == "0.21.0", (
    "Wrong tokenizers version. Restart the runtime after the install cell."
)
assert huggingface_hub.__version__ == "0.28.1", (
    "Wrong huggingface_hub version. Restart the runtime after the install cell."
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Qwen2ForCausalLM,
    BitsAndBytesConfig,
)

print("✓ Qwen2ForCausalLM import succeeded")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print(
        f"VRAM: {props.total_memory / 1024**3:.2f} GB"
    )


Python: 3.13.15
PyTorch: 2.11.0+cu128
Transformers: 4.48.3
Tokenizers: 0.21.0
HF Hub: 0.28.1
CUDA available: True
✓ Qwen2ForCausalLM import succeeded
GPU: Tesla T4
VRAM: 14.56 GB


## 3. Configuration and raw notes


In [3]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
NOTES_FILE = Path("notes_deidentified(1).xlsx")

if not NOTES_FILE.exists():
    fallback = Path("/mnt/data") / NOTES_FILE.name
    if fallback.exists():
        NOTES_FILE = fallback
    else:
        raise FileNotFoundError(
            "Could not find notes_deidentified(1).xlsx. "
            "Place it beside this notebook or update NOTES_FILE."
        )

notes = pd.read_excel(
    NOTES_FILE,
    sheet_name="Query result",
)

notes = notes.rename(
    columns={"Profile Key": "Patient"}
)

required_cols = {"Patient", "Service Date", "Type", "Notes"}
missing = required_cols - set(notes.columns)
if missing:
    raise ValueError(f"Missing required note columns: {sorted(missing)}")

notes["Note_Row_ID"] = np.arange(len(notes))

print("Notes:", len(notes))
print("Patients:", notes["Patient"].nunique())
display(notes[["Patient", "Service Date", "Type", "Note_Row_ID"]].head())

Notes: 215
Patients: 117


,Patient,Service Date,Type,Note_Row_ID
0,Patient_001,2022,Operative Report,0
1,Patient_001,2025,Progress Notes,1
2,Patient_002,2017,Operative Report,2
3,Patient_002,2018,Progress Notes,3
4,Patient_003,2014,Operative Report,4


## 4. Load Qwen2.5-7B-Instruct in 4-bit


In [4]:
tokenizer = None
model = None


def load_qwen_4bit():
    global tokenizer, model

    if model is not None:
        print("Model already loaded.")
        return tokenizer, model

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA GPU not detected."
        )

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    print("Loading tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        use_fast=True,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading Qwen2.5-7B-Instruct...")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )

    model.eval()

    print("✓ Qwen loaded")
    print("Device map:", model.hf_device_map)

    print(
        f"CUDA allocated: "
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
    )

    print(
        f"CUDA reserved:  "
        f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
    )

    return tokenizer, model


tokenizer, model = load_qwen_4bit()


Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✓ Qwen loaded
Device map: {'': 0}
CUDA allocated: 5.17 GB
CUDA reserved:  5.22 GB


## 5. Pass 1 — section-aware high-recall valve retrieval

This stage does **not** interpret the medicine. It only selects potentially relevant text while dropping obvious noise sections such as medications, ROS, social history, and physical examination.

In [5]:
NOISE_HEADINGS = re.compile(
    r"""
    ^family\s+history |
    ^social\s+history |
    ^allerg |
    ^medications? |
    ^current\s+(outpatient\s+)?medications? |
    ^current\s+meds |
    ^review\s+of\s+systems |
    ^physical\s+exam |
    ^physical\s+examination |
    ^objective$ |
    ^education/counseling
    """,
    re.I | re.X,
)

RELEVANT_HEADINGS = re.compile(
    r"""
    history\s+of\s+present\s+illness |
    ^hpi |
    past\s+medical\s+history |
    past\s+surgical\s+history |
    cardiac\s+history |
    cardiac\s+testing |
    cardiovascular\s+medicine\s+testing |
    echocardiogram |
    assessment |
    impression |
    recommendations |
    plan
    """,
    re.I | re.X,
)

STRONG_VALVE_PATTERN = re.compile(
    r"""
    \bAVR\b |
    \bSAVR\b |
    \bTAVR\b |
    \bViV\b |
    valve[- ]in[- ]valve |
    aortic\s+valve |
    prosthetic\s+aortic |
    bioprosthetic |
    prosthetic\s+valve |
    prosthetic\s+(stenosis|regurgitation|failure|dysfunction) |
    paravalvular |
    endocarditis |
    valve\s+thrombosis |
    redo\s+(AVR|aortic) |
    leaflet\s+(thickening|restriction|calcification|degeneration) |
    Carpentier |
    Edwards |
    Evolut |
    Trifecta |
    Magna |
    Perimount |
    Epic |
    Biocor |
    St\.?\s*Jude |
    S3\s+Ultra
    """,
    re.I | re.X,
)

VALVE_MEASUREMENT_PATTERN = re.compile(
    r"""
    peak\s+gradient |
    mean\s+gradient |
    dimensionless\s+(index|valve\s+index) |
    \bAVA\b |
    aortic\s+regurgitation |
    aortic\s+stenosis |
    well[- ]seated |
    stable\s+gradients |
    functioning\s+(well|normally)
    """,
    re.I | re.X,
)


def classify_sections(lines):
    sections = []
    current = "unknown"

    for line in lines:
        s = line.strip()

        if NOISE_HEADINGS.search(s):
            current = "noise"
        elif RELEVANT_HEADINGS.search(s):
            current = "relevant"

        sections.append(current)

    return sections


def select_valve_context(text, context_lines=1):
    lines = [
        x.strip()
        for x in str(text).splitlines()
        if x.strip()
    ]

    sections = classify_sections(lines)
    selected = set()

    for i, line in enumerate(lines):
        if sections[i] == "noise":
            continue

        strong_hit = bool(
            STRONG_VALVE_PATTERN.search(line)
        )

        measurement_hit = bool(
            sections[i] == "relevant"
            and VALVE_MEASUREMENT_PATTERN.search(line)
        )

        if strong_hit or measurement_hit:
            lo = max(0, i - context_lines)
            hi = min(len(lines), i + context_lines + 1)

            for j in range(lo, hi):
                if sections[j] != "noise":
                    selected.add(j)

    # Preserve nearby useful headings.
    for i in list(selected):
        for j in range(max(0, i - 3), i + 1):
            if RELEVANT_HEADINGS.search(lines[j]):
                selected.add(j)

    return "\n".join(
        lines[i]
        for i in sorted(selected)
    )


def token_count(text):
    if tokenizer is None:
        raise RuntimeError(
            "Load the tokenizer first."
        )

    return len(
        tokenizer(
            str(text),
            add_special_tokens=False,
        )["input_ids"]
    )


## 6. Deterministic parser for operative / semi-structured reports

In [6]:
def parse_year(x):
    if x is None:
        return None

    m = re.search(
        r"\b(19\d{2}|20\d{2})\b",
        str(x),
    )
    return int(m.group(1)) if m else None


def parse_number(x):
    if x is None:
        return None

    m = re.search(
        r"\d+(?:\.\d+)?",
        str(x),
    )
    if not m:
        return None

    value = float(m.group())
    return int(value) if value.is_integer() else value


def labelled_value(text, label):
    m = re.search(
        rf"{re.escape(label)}\s*:\s*([^\n\r]+)",
        str(text),
        flags=re.IGNORECASE,
    )
    return m.group(1).strip() if m else None


def parse_operativereport(row):
    text = str(row["Notes"])
    service_year = parse_year(row["Service Date"])

    valve_model = labelled_value(text, "Tissue Implant Type")
    valve_size = labelled_value(text, "Implant Size")
    replacement_type = labelled_value(text, "Replacement Type")
    valve_excision = labelled_value(text, "Valve Excision")
    reoperation = labelled_value(text, "Reoperation")
    valve_in_valve = labelled_value(text, "Valve in Valve")

    low = text.lower()

    if (
        "transcatheter aortic valve" in low
        or "tavr" in low
        or "transfemoral" in low
    ):
        procedure_type = "TAVR"
    elif (
        "aortic valve replacement" in low
        or "valve excision" in low
        or "avr" in low
    ):
        procedure_type = "SAVR"
    else:
        procedure_type = "AVR_unspecified"

    replacement_low = str(replacement_type or "").lower()
    if "tissue" in replacement_low:
        valve_material = "bioprosthetic_tissue"
    elif "mechanical" in replacement_low:
        valve_material = "mechanical"
    else:
        valve_material = "unknown"

    excision_low = str(valve_excision or "").lower()
    if "native" in excision_low:
        target_context = "native_valve"
    elif "prosthe" in excision_low or "previous" in excision_low:
        target_context = "prior_prosthesis"
    else:
        target_context = "unknown"

    is_redo = None
    if reoperation:
        r = reoperation.lower()
        if "no previous" in r or r == "no":
            is_redo = False
        elif "yes" in r:
            is_redo = True

    is_viv = None
    if valve_in_valve:
        v = valve_in_valve.lower()
        if "yes" in v:
            is_viv = True
        elif "no" in v:
            is_viv = False

    concomitant = []
    patterns = [
        (r"\bCABGx?\d*\b", "CABG"),
        (r"\bseptal myectomy\b", "Septal Myectomy"),
        (
            r"\bexcision of anterior mediastinal mass\b",
            "Excision of anterior mediastinal mass",
        ),
        (
            r"\baortic root replacement\b",
            "Aortic root replacement",
        ),
        (
            r"\bascending aort[a-z ]*replacement\b",
            "Ascending aorta replacement",
        ),
    ]

    for pattern, label in patterns:
        if re.search(pattern, text, flags=re.IGNORECASE):
            if label not in concomitant:
                concomitant.append(label)

    evidence_parts = []
    for label in [
        "Replacement Type",
        "Valve Excision",
        "Tissue Implant Type",
        "Implant Size",
        "Valve in Valve",
    ]:
        value = labelled_value(text, label)
        if value is not None:
            evidence_parts.append(f"{label}: {value}")

    procedure = {
        "procedure_year": service_year,
        "procedure_type": procedure_type,
        "target_valve_context": target_context,
        "valve_model": valve_model,
        "valve_size_mm": parse_number(valve_size),
        "valve_material": valve_material,
        "is_redo": is_redo,
        "is_valve_in_valve": is_viv,
        "concomitant_procedures": concomitant,
        "evidence": "; ".join(evidence_parts),
    }

    return {
        "procedures": [procedure],
        "prosthetic_events": [],
        "ambiguities": [],
    }

## 7. Final Qwen-specific extraction prompt

This prompt uses Qwen's chat roles directly: stable extraction rules live in the **system** message, while the real numbered note is the **user** message. The demonstration contains no real valve brands or sizes, reducing example leakage.


In [7]:
QWEN_SYSTEM_PROMPT = """
You are a precise clinical information extractor.

Extract ONLY prosthetic AORTIC-VALVE history from numbered clinical-note lines.
Return exactly ONE valid JSON object and nothing else.
Do not use markdown. Do not explain your reasoning.
Do not guess.

OUTPUT SCHEMA
{
  "procedures": [
    {"year":null,"type":"","model":null,"size":null,"redo":null,"viv":null,"e":null}
  ],
  "events": [
    {"type":"","year":null,"severity":null,"reintervention":null,"e":null}
  ],
  "ambiguities": []
}

Procedure type is exactly one of:
"SAVR", "TAVR", "AVR_unspecified"

Event type is exactly one of:
"stable_function",
"structural_deterioration",
"prosthetic_stenosis",
"prosthetic_regurgitation",
"paravalvular_leak",
"endocarditis",
"thrombosis",
"reintervention",
"other_prosthetic_dysfunction"

RULES

PROCEDURES
- Extract only COMPLETED aortic-valve replacements.
- Planned/scheduled/recommended/considered procedures do not count.
- Historical completed procedures such as "s/p AVR" count.
- A later replacement after a prior replacement has redo=true.
- Set viv=true only for explicit valve-in-valve/ViV OR a TAVR clearly performed in an already-existing surgical bioprosthetic aortic valve.

YEARS
- A year is non-null only when that exact procedure/event is explicitly tied to that year in its cited REAL line.
- [DATE] gives no year.
- Never borrow a year from another line, encounter year, nearby echo, or the example.

MODEL / SIZE
- Copy model and size only when explicitly supported by REAL note lines.
- [NAME] is not a model.
- Never infer a brand/model.

EVENTS
- Events must concern an EXISTING prosthetic aortic valve, not native valve disease before replacement.
- AS / stenosis -> prosthetic_stenosis.
- AI / AR / insufficiency / regurgitation -> prosthetic_regurgitation.
- AI/AR NEVER means stenosis.
- Trace, trivial, or mild AR/AI alone is not clinically meaningful prosthetic_regurgitation.
- Explicit paravalvular/perivalvular leak/regurgitation -> paravalvular_leak.
- Explicit SVD / structural valve deterioration / degeneration -> structural_deterioration.
- Generic prosthetic valve failure/dysfunction with no stated mechanism -> other_prosthetic_dysfunction.
- If a specific mechanism is stated, do not duplicate it as other_prosthetic_dysfunction.
- stable_function requires explicit evidence such as normal/well-functioning/well-seated prosthesis, satisfactory/stable/low gradients, or no significant prosthetic regurgitation.
- Bare gradient numbers alone are not enough.
- Completed repeat AVR/TAVR after prior AVR -> include reintervention.

EVIDENCE
- Every emitted procedure and event MUST have "e" equal to ONE integer line number from the REAL numbered lines.
- The cited line must directly support that fact.
- Never cite a heading-only line such as "Impression", "Assessment", or "Plan".
- If you cannot identify a direct supporting line, OMIT the fact.
- Omit unsupported events entirely.

CONSERVATISM
- Use null rather than inference.
- Do not create facts to fill the schema.
- Multiple observations of the same event type are allowed in the events list; downstream code will consolidate them.

STRUCTURE-ONLY EXAMPLE
The example teaches format only. Never copy its values into the real answer.

Example lines:
1: Prior surgical aortic valve replacement in 2014.
2: Echo in 2021: severe stenosis of the prosthetic aortic valve.
3: Patient underwent successful valve-in-valve TAVR on [DATE].
4: Follow-up: transcatheter aortic prosthesis is well seated and functioning normally.

Example output:
{
  "procedures": [
    {"year":2014,"type":"SAVR","model":null,"size":null,"redo":false,"viv":false,"e":1},
    {"year":null,"type":"TAVR","model":null,"size":null,"redo":true,"viv":true,"e":3}
  ],
  "events": [
    {"type":"prosthetic_stenosis","year":2021,"severity":"severe","reintervention":true,"e":2},
    {"type":"reintervention","year":null,"severity":null,"reintervention":true,"e":3},
    {"type":"stable_function","year":null,"severity":null,"reintervention":null,"e":4}
  ],
  "ambiguities": []
}
"""

QWEN_USER_PROMPT = """
Extract the prosthetic aortic-valve history from the REAL numbered lines below.

Use only facts supported by these lines.
Every emitted fact needs a direct integer evidence line "e".
Return JSON only.

REAL LINES:
{note_text}
"""


## 8. Robust JSON handling and Qwen inference


In [8]:
# ============================================================
# JSON COMPLETION + EXTRACTION
# ============================================================

from transformers import StoppingCriteria, StoppingCriteriaList


def find_complete_json_object(text):
    """
    Return the first complete top-level JSON object in `text`.

    Braces inside quoted JSON strings are ignored. This lets us stop
    Qwen2.5-7B-Instruct as soon as the requested object is complete instead of
    relying on the model to emit an EOS token promptly.
    """

    if text is None:
        return None

    text = str(text)

    start = None
    depth = 0
    in_string = False
    escape = False

    for i, ch in enumerate(text):

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True

        elif ch == "{":
            if start is None:
                start = i
            depth += 1

        elif ch == "}":
            if start is None:
                continue

            depth -= 1

            if depth == 0:
                return text[start:i + 1]

    return None


class StopOnCompleteJSON(StoppingCriteria):
    """
    Stop generation when the first complete top-level JSON object closes.

    This extractor runs with batch size 1, so a single boolean stopping
    decision is appropriate.
    """

    def __init__(
        self,
        tokenizer,
        prompt_length,
    ):
        super().__init__()
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(
        self,
        input_ids,
        scores,
        **kwargs,
    ):
        generated_ids = input_ids[
            0,
            self.prompt_length:
        ]

        text = self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        )

        return (
            find_complete_json_object(text)
            is not None
        )


def extract_json_object(text):
    """
    Parse the first complete top-level JSON object.

    If the object is structurally complete but contains a minor JSON
    formatting error, json-repair is allowed to repair that object only.
    We never repair an unfinished/truncated object.
    """

    if text is None:
        raise ValueError(
            "Model returned empty output"
        )

    candidate = find_complete_json_object(
        text
    )

    if candidate is None:
        raise ValueError(
            "No complete JSON object found"
        )

    try:
        return json.loads(
            candidate
        )

    except json.JSONDecodeError:
        repaired = repair_json(
            candidate,
            return_objects=True,
        )

        if not isinstance(
            repaired,
            dict,
        ):
            raise ValueError(
                "JSON repair did not return an object"
            )

        return repaired


# ============================================================
# NUMBER SELECTED CONTEXT
# ============================================================

def number_context_lines(text):

    lines = [
        line.strip()
        for line in str(text).splitlines()
        if line.strip()
    ]

    numbered_lines = []
    evidence_map = {}

    for i, line in enumerate(
        lines,
        start=1,
    ):

        numbered_lines.append(
            f"{i}:{line}"
        )

        evidence_map[i] = line

    return (
        "\n".join(numbered_lines),
        evidence_map,
    )


# ============================================================
# SINGLE EVIDENCE-ID RESOLVER
# ============================================================

def resolve_evidence_id(
    evidence_id,
    evidence_map,
):
    """
    Resolve ONE evidence line number back
    to the original selected clinical line.
    """

    if evidence_id is None:
        return [], ""

    try:
        evidence_id = int(
            evidence_id
        )

    except (
        TypeError,
        ValueError,
    ):
        return [], ""

    if evidence_id not in evidence_map:
        return [], ""

    return (
        [evidence_id],
        evidence_map[evidence_id],
    )


# ============================================================
# COMPACT OUTPUT -> ORIGINAL/CANONICAL PIPELINE FORMAT
# ============================================================

def compact_to_canonical(
    obj,
    evidence_map,
):
    """
    Convert Qwen's compact JSON into the canonical extraction format.
    Supports the final event-list schema and the older event-dict schema.
    """
    if not isinstance(obj, dict):
        obj = {}

    procedures_out = []
    procedures = obj.get("procedures", [])
    if not isinstance(procedures, list):
        procedures = []

    for p in procedures:
        if not isinstance(p, dict):
            continue

        evidence_ids, evidence_text = resolve_evidence_id(
            p.get("e"),
            evidence_map,
        )

        procedure_type = p.get("type")
        if procedure_type == "AVR":
            procedure_type = "AVR_unspecified"
        if procedure_type not in {
            "TAVR",
            "SAVR",
            "AVR_unspecified",
        }:
            procedure_type = "AVR_unspecified"

        valve_model = p.get("model")
        if isinstance(valve_model, str):
            valve_model = valve_model.strip()
            if (
                "[NAME]" in valve_model
                or valve_model.lower() in {"", "unknown", "none", "null"}
            ):
                valve_model = None

        procedures_out.append(
            {
                "procedure_year": parse_year(p.get("year")),
                "procedure_type": procedure_type,
                "valve_model": valve_model,
                "valve_size_mm": parse_number(p.get("size")),
                "is_redo": p.get("redo"),
                "is_valve_in_valve": p.get("viv"),
                "evidence_ids": evidence_ids,
                "evidence": evidence_text,
            }
        )

    events_out = []
    events = obj.get("events", [])

    # Backward compatibility for old dict-shaped outputs.
    if isinstance(events, dict):
        events = [
            {
                "type": event_type,
                **event_data,
            }
            for event_type, event_data in events.items()
            if isinstance(event_data, dict)
        ]

    if not isinstance(events, list):
        events = []

    allowed_event_types = {
        "stable_function",
        "structural_deterioration",
        "prosthetic_stenosis",
        "prosthetic_regurgitation",
        "paravalvular_leak",
        "endocarditis",
        "thrombosis",
        "reintervention",
        "other_prosthetic_dysfunction",
    }

    for event_data in events:
        if not isinstance(event_data, dict):
            continue

        event_type = event_data.get("type")
        if event_type not in allowed_event_types:
            continue

        evidence_ids, evidence_text = resolve_evidence_id(
            event_data.get("e"),
            evidence_map,
        )

        events_out.append(
            {
                "event_year": parse_year(event_data.get("year")),
                "event_type": event_type,
                "severity": event_data.get("severity"),
                "led_to_reintervention": event_data.get("reintervention"),
                "evidence_ids": evidence_ids,
                "evidence": evidence_text,
            }
        )

    ambiguities = obj.get("ambiguities", [])
    if not isinstance(ambiguities, list):
        ambiguities = []

    return {
        "procedures": procedures_out,
        "prosthetic_events": events_out,
        "ambiguities": ambiguities,
    }


# ============================================================
# DETERMINISTIC EVIDENCE VALIDATION
# ============================================================

YEAR_RE = re.compile(r"\b(19\d{2}|20\d{2})\b", re.I)

PROCEDURE_COMPLETION_RE = re.compile(
    r"\b("
    r"underwent|status\s+post|s/p|"
    r"implant(?:ed|ation)?|replacement|"
    r"TAVR|SAVR|AVR"
    r")\b",
    re.I,
)

IMAGING_RE = re.compile(
    r"\b(echo|echocardiogram|TTE|TEE|gradient|LVEF|EF)\b",
    re.I,
)

TAVR_RE = re.compile(
    r"\b(TAVR|transcatheter\s+aortic\s+valve)\b",
    re.I,
)

SAVR_RE = re.compile(
    r"\b(SAVR|surgical\s+aortic\s+valve|"
    r"bioprosthetic\s+AVR|aortic\s+valve\s+replacement|\bAVR\b)\b",
    re.I,
)

STENOSIS_RE = re.compile(
    r"\b(stenos(?:is|ed)?|AS)\b",
    re.I,
)

REGURG_RE = re.compile(
    r"\b(regurg(?:itation|itant)?|insufficien(?:cy|t)?|AI|AR)\b",
    re.I,
)

PVL_RE = re.compile(
    r"\b(paravalv\w*|perivalv\w*)\b",
    re.I,
)

ENDO_RE = re.compile(r"\bendocard\w*\b", re.I)
THROMB_RE = re.compile(r"\bthromb\w*\b", re.I)
SVD_RE = re.compile(
    r"\b(SVD|structural\s+valve\s+deterioration|degenerat\w*)\b",
    re.I,
)
FAILURE_RE = re.compile(r"\b(failure|dysfunction)\b", re.I)

STABLE_RE = re.compile(
    r"\b("
    r"well[-\s]?seated|well[-\s]?functioning|"
    r"functioning\s+(?:normally|well)|"
    r"normal\s+(?:prosthetic|prosthesis|valve)\w*|"
    r"stable\s+gradient|low\s+gradient|"
    r"no\s+(?:significant\s+)?(?:aortic\s+)?regurgitation|"
    r"no\s+(?:significant\s+)?(?:aortic\s+)?insufficiency"
    r")\b",
    re.I,
)

LOW_GRADE_REGURG_RE = re.compile(
    r"\b(trace|trivial|mild|1\+)\b",
    re.I,
)

MEANINGFUL_REGURG_RE = re.compile(
    r"\b(moderate|severe|2\+|3\+|4\+)\b",
    re.I,
)

HEADING_ONLY_RE = re.compile(
    r"^\s*(impression|assessment|plan|history|exam|diagnosis|diagnoses)\s*:?\s*$",
    re.I,
)


def _explicit_year_from_evidence(text):
    if not text:
        return None
    years = YEAR_RE.findall(str(text))
    years = [int(y) for y in years]
    if len(set(years)) == 1:
        return years[0]
    return None


def _value_explicit_in_evidence(value, text):
    if value is None or not text:
        return False
    return str(value).lower() in str(text).lower()


def _clean_model(model, evidence):
    """
    Keep a model only when it is textually supported by its cited line.
    Remove size fragments such as '#27' from the model field.
    """
    if not isinstance(model, str) or not evidence:
        return None

    model = re.sub(r"\s*#?\s*\d{2}\s*(?:mm)?\s*$", "", model).strip()
    if not model:
        return None

    ev = re.sub(r"[^a-z0-9]+", " ", evidence.lower()).strip()
    mod = re.sub(r"[^a-z0-9]+", " ", model.lower()).strip()

    # Exact normalized phrase.
    if mod in ev:
        return model

    # Allow tiny spelling variation such as Biocor/Biocore.
    import difflib
    ev_tokens = ev.split()
    for token in mod.split():
        if len(token) < 4:
            continue
        if any(
            difflib.SequenceMatcher(None, token, et).ratio() >= 0.86
            for et in ev_tokens
        ):
            continue
        return None

    return model


def _validated_size(size, evidence):
    if size is None or not evidence:
        return None

    try:
        size_num = int(float(size))
    except Exception:
        return None

    # Require the claimed number to appear in valve-size-like syntax.
    patterns = [
        rf"#\s*{size_num}\b",
        rf"\b{size_num}\s*mm\b",
        rf"\bsize\s*#?\s*{size_num}\b",
    ]

    if any(re.search(p, evidence, re.I) for p in patterns):
        return size_num

    return None


def _procedure_evidence_valid(proc_type, evidence):
    if not evidence or HEADING_ONLY_RE.match(evidence):
        return False

    low = evidence.lower()

    if proc_type == "TAVR":
        type_ok = bool(TAVR_RE.search(evidence))
    elif proc_type == "SAVR":
        type_ok = bool(SAVR_RE.search(evidence))
    else:
        type_ok = bool(SAVR_RE.search(evidence) or TAVR_RE.search(evidence))

    if not type_ok:
        return False

    # Echo-only mention of an existing prosthesis is not a procedure.
    if IMAGING_RE.search(evidence):
        strong_completion = re.search(
            r"\b(underwent|s/p|status\s+post|replacement|implanted|implantation)\b",
            evidence,
            re.I,
        )
        if not strong_completion:
            return False

    return bool(PROCEDURE_COMPLETION_RE.search(evidence))


def _event_evidence_valid(event_type, evidence):
    if not evidence or HEADING_ONLY_RE.match(evidence):
        return False

    if event_type == "prosthetic_stenosis":
        return bool(STENOSIS_RE.search(evidence))

    if event_type == "prosthetic_regurgitation":
        if not REGURG_RE.search(evidence):
            return False
        if PVL_RE.search(evidence):
            # PVL can coexist with regurgitation, but low-grade AI alone is not enough.
            return bool(MEANINGFUL_REGURG_RE.search(evidence))
        if LOW_GRADE_REGURG_RE.search(evidence) and not MEANINGFUL_REGURG_RE.search(evidence):
            return False
        return True

    if event_type == "paravalvular_leak":
        return bool(PVL_RE.search(evidence))

    if event_type == "endocarditis":
        return bool(ENDO_RE.search(evidence))

    if event_type == "thrombosis":
        return bool(THROMB_RE.search(evidence))

    if event_type == "structural_deterioration":
        return bool(SVD_RE.search(evidence))

    if event_type == "other_prosthetic_dysfunction":
        if not FAILURE_RE.search(evidence):
            return False
        # Do not keep generic dysfunction when a specific mechanism is explicit.
        if any(
            rex.search(evidence)
            for rex in [STENOSIS_RE, REGURG_RE, PVL_RE, ENDO_RE, THROMB_RE, SVD_RE]
        ):
            return False
        return True

    if event_type == "stable_function":
        if not STABLE_RE.search(evidence):
            return False
        # A line explicitly calling the prosthesis moderate/severe stenotic is not
        # accepted as stable function solely because other values are present.
        if STENOSIS_RE.search(evidence) and re.search(r"\b(moderate|severe)\b", evidence, re.I):
            return False
        return True

    if event_type == "reintervention":
        return bool(
            (TAVR_RE.search(evidence) or SAVR_RE.search(evidence))
            and re.search(
                r"\b(underwent|s/p|status\s+post|replacement|implant(?:ed|ation)?)\b",
                evidence,
                re.I,
            )
        )

    return False


def validate_canonical_extraction(obj):
    """
    Deterministic guardrails.
    Qwen proposes facts; Python decides whether the cited evidence supports them.

    This intentionally favors precision over filling every field.
    """
    if not isinstance(obj, dict):
        return {
            "procedures": [],
            "prosthetic_events": [],
            "ambiguities": [],
        }

    validated_procedures = []

    for p in obj.get("procedures", []):
        if not isinstance(p, dict):
            continue

        evidence = str(p.get("evidence") or "")
        proc_type = p.get("procedure_type")

        if not p.get("evidence_ids"):
            continue

        if not _procedure_evidence_valid(proc_type, evidence):
            continue

        q = dict(p)

        # Chronology must come from the cited line only.
        q["procedure_year"] = _explicit_year_from_evidence(evidence)

        q["valve_size_mm"] = _validated_size(
            q.get("valve_size_mm"),
            evidence,
        )

        q["valve_model"] = _clean_model(
            q.get("valve_model"),
            evidence,
        )

        validated_procedures.append(q)

    # Deterministic redo/ViV cleanup.
    for i, p in enumerate(validated_procedures):
        if i == 0:
            if p.get("is_redo") is not True:
                p["is_redo"] = False
            if p.get("procedure_type") != "TAVR":
                p["is_valve_in_valve"] = False
            continue

        # Every subsequent completed AVR is a redo.
        p["is_redo"] = True

        if p.get("procedure_type") == "TAVR":
            prior = validated_procedures[:i]
            prior_bioprosthetic = any(
                re.search(
                    r"\b(bioprosth\w*|tissue|pericardial)\b",
                    str(x.get("evidence") or ""),
                    re.I,
                )
                for x in prior
            )
            explicit_viv = bool(
                re.search(
                    r"\b(valve[-\s]?in[-\s]?valve|ViV)\b",
                    str(p.get("evidence") or ""),
                    re.I,
                )
            )
            if explicit_viv or prior_bioprosthetic:
                p["is_valve_in_valve"] = True

    validated_events = []

    for e in obj.get("prosthetic_events", []):
        if not isinstance(e, dict):
            continue

        evidence = str(e.get("evidence") or "")
        event_type = e.get("event_type")

        if not e.get("evidence_ids"):
            continue

        if not _event_evidence_valid(event_type, evidence):
            continue

        q = dict(e)
        q["event_year"] = _explicit_year_from_evidence(evidence)
        validated_events.append(q)

    # If we have >1 validated completed valve procedures, create/repair a
    # reintervention event from the later procedure itself.
    if len(validated_procedures) > 1:
        later = validated_procedures[-1]

        if later.get("evidence_ids"):
            validated_events.append(
                {
                    "event_year": later.get("procedure_year"),
                    "event_type": "reintervention",
                    "severity": None,
                    "led_to_reintervention": True,
                    "evidence_ids": later.get("evidence_ids"),
                    "evidence": later.get("evidence"),
                }
            )

    # Deduplicate event type using the first validated direct evidence.
    # Prefer more clinically severe wording when duplicates exist.
    severity_rank = {
        None: 0,
        "trace": 1,
        "trivial": 1,
        "mild": 2,
        "moderate": 3,
        "severe": 4,
    }

    by_type = {}

    for e in validated_events:
        t = e["event_type"]
        sev = str(e.get("severity") or "").lower()

        score = 0
        for key, rank in severity_rank.items():
            if key is not None and key in sev:
                score = max(score, rank)

        if t not in by_type or score > by_type[t][0]:
            by_type[t] = (score, e)

    validated_events = [
        pair[1]
        for pair in by_type.values()
    ]

    return {
        "procedures": validated_procedures,
        "prosthetic_events": validated_events,
        "ambiguities": obj.get("ambiguities", [])
        if isinstance(obj.get("ambiguities", []), list)
        else [],
    }


# ============================================================
# QWEN FOLLOW-UP EXTRACTION
# ============================================================

def qwen_extract_followup(
    row,
    verbose=True,
):

    full_note_text = str(
        row["Notes"]
    )

    # --------------------------------------------------------
    # PASS 1:
    # deterministic high-recall valve context retrieval
    # --------------------------------------------------------

    selected_text = (
        select_valve_context(
            full_note_text,
            context_lines=1,
        )
    )

    # --------------------------------------------------------
    # Number the selected lines.
    # Qwen returns ONE evidence line number per fact.
    # --------------------------------------------------------

    (
        note_text,
        evidence_map,
    ) = number_context_lines(
        selected_text
    )

    # --------------------------------------------------------
    # QWEN SYSTEM + USER PROMPT
    #
    # Intentionally NO service year: avoid chronology leakage.
    # --------------------------------------------------------

    user_prompt = QWEN_USER_PROMPT.format(
        note_text=note_text,
    )

    messages = [
        {
            "role": "system",
            "content": QWEN_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    # --------------------------------------------------------
    # GPU cleanup
    # --------------------------------------------------------

    gc.collect()
    torch.cuda.empty_cache()

    text_device = next(
        model.parameters()
    ).device

    inputs = {
        k: (
            v.to(
                text_device
            )
            if torch.is_tensor(v)
            else v
        )
        for (
            k,
            v,
        ) in inputs.items()
    }

    input_len = (
        inputs[
            "input_ids"
        ].shape[-1]
    )

    if verbose:

        print(
            f"Context: "
            f"{token_count(full_note_text)} "
            f"→ "
            f"{token_count(selected_text)} "
            f"tokens"
        )

        print(
            f"Actual Qwen prompt: "
            f"{input_len} tokens"
        )

    # --------------------------------------------------------
    # GENERATION
    #
    # MAX_NEW_TOKENS is only a safety ceiling. In normal operation,
    # StopOnCompleteJSON ends generation as soon as the requested
    # top-level JSON object closes.
    # --------------------------------------------------------

    MAX_NEW_TOKENS = 1000

    json_stopper = StopOnCompleteJSON(
        tokenizer=tokenizer,
        prompt_length=input_len,
    )

    with torch.inference_mode():

        output_ids = (
            model.generate(
                **inputs,
                max_new_tokens=(
                    MAX_NEW_TOKENS
                ),
                do_sample=False,
                use_cache=True,
                repetition_penalty=1.05,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                stopping_criteria=(
                    StoppingCriteriaList(
                        [json_stopper]
                    )
                ),
            )
        )

    generated = output_ids[
        0,
        input_len:
    ]

    generated_n = len(
        generated
    )

    if verbose:

        print(
            f"Generated tokens: "
            f"{generated_n} / "
            f"{MAX_NEW_TOKENS}"
        )

    raw_text = (
        tokenizer.decode(
            generated,
            skip_special_tokens=True,
        )
    )

    # --------------------------------------------------------
    # PARSING
    #
    # Reaching MAX_NEW_TOKENS is not itself a failure. If a complete
    # JSON object exists, we keep it. A ceiling hit is a failure only
    # when no complete top-level object was produced.
    # --------------------------------------------------------

    parse_error = None

    try:

        compact_obj = (
            extract_json_object(
                raw_text
            )
        )

        obj = compact_to_canonical(
            compact_obj,
            evidence_map,
        )

        obj = validate_canonical_extraction(
            obj
        )

    except Exception as e:

        if (
            generated_n
            >= MAX_NEW_TOKENS
        ):
            parse_error = (
                "Generation reached "
                f"{MAX_NEW_TOKENS} tokens "
                "without completing JSON"
            )

        else:
            parse_error = repr(
                e
            )

        obj = {}

    # --------------------------------------------------------
    # GUARANTEE CANONICAL TOP-LEVEL STRUCTURE
    # --------------------------------------------------------

    if not isinstance(
        obj,
        dict,
    ):

        parse_error = (
            parse_error
            or
            f"Parsed output is "
            f"{type(obj).__name__}, "
            f"not dict"
        )

        obj = {}

    obj.setdefault(
        "procedures",
        [],
    )

    obj.setdefault(
        "prosthetic_events",
        [],
    )

    obj.setdefault(
        "ambiguities",
        [],
    )

    # --------------------------------------------------------
    # RETURN
    # --------------------------------------------------------

    return {
        "result": (
            obj
        ),

        "raw_text": (
            raw_text
        ),

        # Unnumbered selected text
        "selected_context": (
            selected_text
        ),

        # Exactly what Qwen2.5-7B-Instruct saw
        "numbered_context": (
            note_text
        ),

        # line number -> original line
        "evidence_map": (
            evidence_map
        ),

        "parse_error": (
            parse_error
        ),

        "prompt_tokens": (
            input_len
        ),

        "generated_tokens": (
            generated_n
        ),
    }


# ============================================================
# DISPATCHER
# ============================================================

def extract_valve_note(
    row,
    verbose=True,
):

    note_type = str(
        row["Type"]
    ).lower()

    # --------------------------------------------------------
    # Structured operative / procedure notes:
    # deterministic parser ONLY.
    #
    # Qwen is NOT called.
    # --------------------------------------------------------

    if (
        "operative"
        in note_type

        or

        "procedure"
        in note_type
    ):

        return {
            "result": (
                parse_operativereport(
                    row
                )
            ),

            "raw_text": None,

            "selected_context": (
                str(
                    row["Notes"]
                )
            ),

            "numbered_context": None,

            "evidence_map": None,

            "parse_error": None,

            "prompt_tokens": None,

            "generated_tokens": None,

            "mode": (
                "deterministic"
            ),
        }

    # --------------------------------------------------------
    # Narrative notes:
    #
    # Pass 1 selector
    #       ↓
    # numbered evidence
    #       ↓
    # Qwen
    # --------------------------------------------------------

    out = (
        qwen_extract_followup(
            row,
            verbose=verbose,
        )
    )

    out[
        "mode"
    ] = "qwen2.5_7b"

    return out

## 9. Frozen 10-note benchmark

These are the 10 manually adjudicated notes. The rows are reconstructed directly from patient + service year + note type; if more than one row matches, the longest note is selected, mirroring the original benchmark construction.

In [9]:
BENCHMARK_CASES = [
    ("B01", "Patient_035", 2024, "Progress Notes"),
    ("B02", "Patient_042", 2023, "Progress Notes"),
    ("B03", "Patient_086", 2025, "Progress Notes"),
    ("B04", "Patient_104", 2022, "Progress Notes"),
    ("B05", "Patient_068", 2024, "Progress Notes"),
    ("B06", "Patient_044", 2024, "Progress Notes"),
    ("B07", "Patient_058", 2024, "Progress Notes"),
    ("B08", "Patient_038", 2024, "Progress Notes"),
    ("B09", "Patient_036", 2024, "Progress Notes"),
    ("B10", "Patient_103", 2023, "Progress Notes"),
]


def locate_benchmark_note(patient, year, note_type):
    service_year = pd.to_numeric(
        notes["Service Date"],
        errors="coerce",
    )

    pool = notes[
        notes["Patient"].eq(patient)
        & service_year.eq(year)
        & notes["Type"].eq(note_type)
    ].copy()

    if len(pool) == 0:
        raise ValueError(
            f"No benchmark row found for "
            f"{patient}, {year}, {note_type}"
        )

    if len(pool) > 1:
        print(
            f"Warning: {len(pool)} matches for {patient}; "
            f"using longest note."
        )

    pool["_note_len"] = (
        pool["Notes"].astype(str).str.len()
    )

    return (
        pool.sort_values("_note_len", ascending=False)
        .iloc[0]
        .drop(labels=["_note_len"])
    )


rows = []
for benchmark_id, patient, year, note_type in BENCHMARK_CASES:
    row = locate_benchmark_note(
        patient,
        year,
        note_type,
    ).copy()
    row["Benchmark_ID"] = benchmark_id
    rows.append(row)

benchmark_gold_10 = pd.DataFrame(rows).reset_index(drop=True)

display(
    benchmark_gold_10[
        ["Benchmark_ID", "Patient", "Service Date", "Type", "Note_Row_ID"]
    ]
)

,Benchmark_ID,Patient,Service Date,Type,Note_Row_ID
0,B01,Patient_035,2024,Progress Notes,113
1,B02,Patient_042,2023,Progress Notes,76
2,B03,Patient_086,2025,Progress Notes,167
3,B04,Patient_104,2022,Progress Notes,201
4,B05,Patient_068,2024,Progress Notes,133
5,B06,Patient_044,2024,Progress Notes,80
6,B07,Patient_058,2024,Progress Notes,107
7,B08,Patient_038,2024,Progress Notes,70
8,B09,Patient_036,2024,Progress Notes,66
9,B10,Patient_103,2023,Progress Notes,200


### Human gold labels

In [10]:
GOLD = {
    "B01": {
        "gold_procedure_types": ["SAVR", "TAVR"],
        "gold_procedure_years": [2010, None],
        "gold_valve_sizes": [27, 26],
        "gold_is_valve_in_valve": True,
        "gold_is_redo": True,
        "gold_event_types": [
            "prosthetic_stenosis",
            "prosthetic_regurgitation",
            "paravalvular_leak",
            "reintervention",
            "stable_function",
        ],
        "gold_has_prosthetic_failure": True,
        "gold_has_reintervention": True,
        "gold_has_stable_function": True,
        "gold_should_be_empty": False,
        "gold_notes": (
            "Biocor #27 bioprosthetic SAVR in 2010; later developed "
            "prosthetic stenosis and regurgitation with perivalvular component, "
            "followed by 26-mm TAVR. Recent TAVR gradients are stable."
        ),
    },

    "B02": {
        "gold_procedure_types": ["SAVR"],
        "gold_procedure_years": [2017],
        "gold_valve_sizes": [25],
        "gold_is_valve_in_valve": False,
        "gold_is_redo": False,
        "gold_event_types": [
            "prosthetic_stenosis",
            "stable_function",
        ],
        "gold_has_prosthetic_failure": False,
        "gold_has_reintervention": False,
        "gold_has_stable_function": True,
        "gold_should_be_empty": False,
        "gold_notes": (
            "2017 SAVR with Trifecta #25. Moderate prosthetic aortic "
            "stenosis was documented and treated medically; later echo "
            "described the AVR as functioning well / stenosis improved."
        ),
    },

    "B03": {
        "gold_procedure_types": ["SAVR"],
        "gold_procedure_years": [2016],
        "gold_valve_sizes": [25],
        "gold_is_valve_in_valve": False,
        "gold_is_redo": False,
        "gold_event_types": ["stable_function"],
        "gold_has_prosthetic_failure": False,
        "gold_has_reintervention": False,
        "gold_has_stable_function": True,
        "gold_should_be_empty": False,
        "gold_notes": (
            "Severe stenosis refers to the native valve before the 2016 "
            "bioprosthetic AVR. Later Carpentier-Edwards #25 prosthesis "
            "is described as functioning normally."
        ),
    },

    "B04": {
        "gold_procedure_types": ["SAVR"],
        "gold_procedure_years": [None],
        "gold_valve_sizes": [25],
        "gold_is_valve_in_valve": False,
        "gold_is_redo": False,
        "gold_event_types": [
            "prosthetic_regurgitation",
            "paravalvular_leak",
        ],
        "gold_has_prosthetic_failure": True,
        "gold_has_reintervention": False,
        "gold_has_stable_function": False,
        "gold_should_be_empty": False,
        "gold_notes": (
            "Existing Carpentier-Edwards #25 prosthesis with severe "
            "paravalvular AR/degeneration. TAVR/PVL repair is planned, "
            "not documented as completed."
        ),
    },

    "B05": {
        "gold_procedure_types": ["SAVR", "TAVR"],
        "gold_procedure_years": [None, None],
        "gold_valve_sizes": [21, 23],
        "gold_is_valve_in_valve": True,
        "gold_is_redo": True,
        "gold_event_types": [
            "prosthetic_stenosis",
            "reintervention",
            "stable_function",
        ],
        "gold_has_prosthetic_failure": True,
        "gold_has_reintervention": True,
        "gold_has_stable_function": True,
        "gold_should_be_empty": False,
        "gold_notes": (
            "Prior #21 surgical pericardial AVR developed severe prosthetic "
            "stenosis from thickening/calcification, followed by successful "
            "23-mm valve-in-valve TAVR."
        ),
    },

    "B06": {
        "gold_procedure_types": ["SAVR", "TAVR"],
        "gold_procedure_years": [2009, 2023],
        "gold_valve_sizes": [25, 26],
        "gold_is_valve_in_valve": True,
        "gold_is_redo": True,
        "gold_event_types": [
            "other_prosthetic_dysfunction",
            "reintervention",
            "stable_function",
        ],
        "gold_has_prosthetic_failure": True,
        "gold_has_reintervention": True,
        "gold_has_stable_function": True,
        "gold_should_be_empty": False,
        "gold_notes": (
            "25-mm surgical bioprosthetic AVR in 2009. In 12/2023, "
            "bioprosthetic valve failure caused cardiogenic shock, followed "
            "by emergent 26-mm ViV TAVR. Magna CE vs Magna Ease conflicts."
        ),
    },

    "B07": {
        "gold_procedure_types": ["SAVR", "TAVR"],
        "gold_procedure_years": [None, 2024],
        "gold_valve_sizes": [27, 26],
        "gold_is_valve_in_valve": True,
        "gold_is_redo": True,
        "gold_event_types": [
            "prosthetic_stenosis",
            "reintervention",
            "stable_function",
        ],
        "gold_has_prosthetic_failure": True,
        "gold_has_reintervention": True,
        "gold_has_stable_function": True,
        "gold_should_be_empty": False,
        "gold_notes": (
            "Prior Trifecta GT #27 surgical AVR developed progressive "
            "prosthetic stenosis. Underwent 26-mm ViV TAVR in 08/2024; "
            "subsequent valve is stable and well seated."
        ),
    },

    "B08": {
        "gold_procedure_types": ["SAVR", "TAVR"],
        "gold_procedure_years": [None, 2023],
        "gold_valve_sizes": [25, 26],
        "gold_is_valve_in_valve": True,
        "gold_is_redo": True,
        "gold_event_types": [
            "prosthetic_stenosis",
            "reintervention",
            "stable_function",
        ],
        "gold_has_prosthetic_failure": True,
        "gold_has_reintervention": True,
        "gold_has_stable_function": True,
        "gold_should_be_empty": False,
        "gold_notes": (
            "Prior CE Perimount #25 surgical AVR developed severe "
            "bioprosthetic stenosis and underwent ViV TAVR 12/2023 "
            "with 26-mm Edwards S3 Ultra."
        ),
    },

    "B09": {
        "gold_procedure_types": ["SAVR", "TAVR"],
        "gold_procedure_years": [2010, None],
        "gold_valve_sizes": [25, 26],
        "gold_is_valve_in_valve": True,
        "gold_is_redo": True,
        "gold_event_types": [
            "reintervention",
            "stable_function",
        ],
        "gold_has_prosthetic_failure": False,
        "gold_has_reintervention": True,
        "gold_has_stable_function": True,
        "gold_should_be_empty": False,
        "gold_notes": (
            "2010 #25 St. Jude Biocor SAVR followed by completed "
            "26-mm Evolut FX ViV TAVR. Mechanism requiring reintervention "
            "is not explicitly stated."
        ),
    },

    "B10": {
        "gold_procedure_types": ["TAVR", "SAVR"],
        "gold_procedure_years": [None, 2023],
        "gold_valve_sizes": [None, 23],
        "gold_is_valve_in_valve": False,
        "gold_is_redo": True,
        "gold_event_types": [
            "endocarditis",
            "reintervention",
            "stable_function",
        ],
        "gold_has_prosthetic_failure": True,
        "gold_has_reintervention": True,
        "gold_has_stable_function": True,
        "gold_should_be_empty": False,
        "gold_notes": (
            "Prior TAVR developed prosthetic valve endocarditis, followed "
            "by redo SAVR with #23 Epic in 02/2023. Original TAVR year "
            "conflicts: 2019 vs 01/2020."
        ),
    },
}

for key in next(iter(GOLD.values())).keys():
    benchmark_gold_10[key] = benchmark_gold_10["Benchmark_ID"].map(
        lambda bid: GOLD[bid][key]
    )

display(
    benchmark_gold_10[
        [
            "Benchmark_ID",
            "Patient",
            "gold_procedure_types",
            "gold_event_types",
            "gold_has_prosthetic_failure",
        ]
    ]
)

,Benchmark_ID,Patient,gold_procedure_types,gold_event_types,gold_has_prosthetic_failure
0,B01,Patient_035,"[SAVR, TAVR]","[prosthetic_stenosis, prosthetic_regurgitation...",True
1,B02,Patient_042,[SAVR],"[prosthetic_stenosis, stable_function]",False
2,B03,Patient_086,[SAVR],[stable_function],False
3,B04,Patient_104,[SAVR],"[prosthetic_regurgitation, paravalvular_leak]",True
4,B05,Patient_068,"[SAVR, TAVR]","[prosthetic_stenosis, reintervention, stable_f...",True
5,B06,Patient_044,"[SAVR, TAVR]","[other_prosthetic_dysfunction, reintervention,...",True
6,B07,Patient_058,"[SAVR, TAVR]","[prosthetic_stenosis, reintervention, stable_f...",True
7,B08,Patient_038,"[SAVR, TAVR]","[prosthetic_stenosis, reintervention, stable_f...",True
8,B09,Patient_036,"[SAVR, TAVR]","[reintervention, stable_function]",False
9,B10,Patient_103,"[TAVR, SAVR]","[endocarditis, reintervention, stable_function]",True


## 10. Audit Pass 1 before model inference

This is a quick safety check that the selector preserves the clinically relevant chronology while reducing token load.

In [11]:
audit_rows = []

for _, row in benchmark_gold_10.iterrows():
    full = str(row["Notes"])
    selected = select_valve_context(full)

    audit_rows.append(
        {
            "Benchmark_ID": row["Benchmark_ID"],
            "Patient": row["Patient"],
            "full_tokens": token_count(full),
            "selected_tokens": token_count(selected),
            "retained_pct": round(
                100 * token_count(selected) / token_count(full),
                1,
            ),
        }
    )

context_audit = pd.DataFrame(audit_rows)
display(context_audit)

,Benchmark_ID,Patient,full_tokens,selected_tokens,retained_pct
0,B01,Patient_035,5339,2116,39.6
1,B02,Patient_042,4877,1235,25.3
2,B03,Patient_086,3440,916,26.6
3,B04,Patient_104,3072,697,22.7
4,B05,Patient_068,3057,737,24.1
5,B06,Patient_044,1584,675,42.6
6,B07,Patient_058,4422,1389,31.4
7,B08,Patient_038,4086,825,20.2
8,B09,Patient_036,2632,609,23.1
9,B10,Patient_103,3196,384,12.0


## 11. One-note smoke test

Run B01 once before the full benchmark. A JSON parse failure will now be returned as metadata instead of crashing the cell.

In [18]:
# ============================================================
# FINAL IN-PLACE QWEN PATCH
# multi-line evidence + deterministic re-grounding
# ============================================================

import re
import difflib


# ------------------------------------------------------------
# 1. FINAL QWEN PROMPT
# ------------------------------------------------------------

QWEN_SYSTEM_PROMPT = """
You are a precise clinical information extractor.

Extract ONLY prosthetic AORTIC-VALVE history from numbered clinical-note lines.

Return exactly ONE valid JSON object and nothing else.
Do not use markdown.
Do not explain your reasoning.
Do not guess.

OUTPUT SCHEMA
{
  "procedures": [
    {
      "year": null,
      "type": "",
      "model": null,
      "size": null,
      "redo": null,
      "viv": null,
      "e": []
    }
  ],
  "events": [
    {
      "type": "",
      "year": null,
      "severity": null,
      "reintervention": null,
      "e": []
    }
  ],
  "ambiguities": []
}

Procedure type:
"SAVR", "TAVR", or "AVR_unspecified"

Event type:
"stable_function",
"structural_deterioration",
"prosthetic_stenosis",
"prosthetic_regurgitation",
"paravalvular_leak",
"endocarditis",
"thrombosis",
"reintervention",
"other_prosthetic_dysfunction"

RULES

PROCEDURES
- Extract only COMPLETED aortic-valve replacements.
- Planned/scheduled/recommended procedures do not count.
- Historical completed procedures such as "s/p AVR" count.
- A later valve replacement after a prior valve replacement has redo=true.
- viv=true for explicit valve-in-valve/ViV OR a TAVR clearly performed in an existing surgical bioprosthetic aortic valve.

YEARS
- A year is non-null only when that procedure/event is explicitly tied to the year in the REAL note.
- [DATE] gives no year.
- Never borrow years from another event, echo, encounter date, or example.

MODEL / SIZE
- Copy only values supported somewhere in the REAL lines.
- Different evidence lines may support procedure date, model, and size.
- [NAME] is not a model.
- Never infer a brand.

EVENTS
- Events must concern an EXISTING prosthetic aortic valve.
- AS / stenosis = prosthetic_stenosis.
- AI / AR / insufficiency / regurgitation = prosthetic_regurgitation.
- AI/AR NEVER means stenosis.
- Trace/trivial/mild AR alone is not clinically meaningful prosthetic regurgitation.
- Explicit paravalvular/perivalvular leak/regurgitation = paravalvular_leak.
- Explicit SVD / structural deterioration / degeneration = structural_deterioration.
- Generic prosthetic failure/dysfunction without a mechanism = other_prosthetic_dysfunction.
- Do not additionally call a specific stenosis/regurgitation/PVL finding generic dysfunction.
- stable_function requires evidence of a well-functioning/well-seated prosthesis, satisfactory/stable/low gradients, or no significant AR.
- A completed repeat AVR/TAVR after prior AVR = reintervention.

EVIDENCE
- "e" is a LIST containing 1 to 3 integer evidence-line IDs.
- Use multiple lines when different parts of one fact are documented separately.
- Never cite heading-only lines such as "Impression" or "Assessment".
- Every cited line must support some part of the fact.

CONSERVATISM
- Use null rather than inference.
- Omit unsupported events.
- Do not invent values to fill the schema.

STRUCTURE-ONLY EXAMPLE

Example lines:
1: Prior surgical aortic valve replacement in 2014.
2: Echo in 2021: severe stenosis of the prosthetic aortic valve.
3: Patient underwent successful valve-in-valve TAVR on [DATE].
4: Follow-up: transcatheter aortic prosthesis is well seated and functioning normally.

Example output:
{
  "procedures": [
    {
      "year": 2014,
      "type": "SAVR",
      "model": null,
      "size": null,
      "redo": false,
      "viv": false,
      "e": [1]
    },
    {
      "year": null,
      "type": "TAVR",
      "model": null,
      "size": null,
      "redo": true,
      "viv": true,
      "e": [3]
    }
  ],
  "events": [
    {
      "type": "prosthetic_stenosis",
      "year": 2021,
      "severity": "severe",
      "reintervention": true,
      "e": [2]
    },
    {
      "type": "reintervention",
      "year": null,
      "severity": null,
      "reintervention": true,
      "e": [3]
    },
    {
      "type": "stable_function",
      "year": null,
      "severity": null,
      "reintervention": null,
      "e": [4]
    }
  ],
  "ambiguities": []
}
"""


QWEN_USER_PROMPT = """
Extract the prosthetic aortic-valve history from the REAL numbered lines below.

Use only the REAL lines.
Every emitted fact must contain 1-3 supporting evidence IDs in "e".

Return JSON only.

REAL LINES:
{note_text}
"""


# ------------------------------------------------------------
# 2. MULTI-EVIDENCE HANDLING
# ------------------------------------------------------------

_LAST_EVIDENCE_MAP = {}


def resolve_evidence_ids(e, evidence_map):

    if e is None:
        return [], ""

    if not isinstance(e, (list, tuple)):
        e = [e]

    ids = []

    for x in e:
        try:
            x = int(x)
        except (TypeError, ValueError):
            continue

        if x in evidence_map and x not in ids:
            ids.append(x)

    text = " | ".join(
        evidence_map[x]
        for x in ids
    )

    return ids, text


def compact_to_canonical(obj, evidence_map):
    """
    Override the previous converter.

    Also stores the current evidence map so the existing
    qwen_extract_followup() function does not need editing.
    """

    global _LAST_EVIDENCE_MAP
    _LAST_EVIDENCE_MAP = evidence_map

    if not isinstance(obj, dict):
        obj = {}

    procedures_out = []

    for p in obj.get("procedures", []):

        if not isinstance(p, dict):
            continue

        ids, text = resolve_evidence_ids(
            p.get("e"),
            evidence_map,
        )

        procedure_type = p.get("type")

        if procedure_type == "AVR":
            procedure_type = "AVR_unspecified"

        if procedure_type not in {
            "SAVR",
            "TAVR",
            "AVR_unspecified",
        }:
            procedure_type = "AVR_unspecified"

        procedures_out.append(
            {
                "procedure_year": parse_year(
                    p.get("year")
                ),
                "procedure_type": procedure_type,
                "valve_model": p.get("model"),
                "valve_size_mm": parse_number(
                    p.get("size")
                ),
                "is_redo": p.get("redo"),
                "is_valve_in_valve": p.get("viv"),
                "evidence_ids": ids,
                "evidence": text,
            }
        )

    events = obj.get("events", [])

    # backwards compatibility
    if isinstance(events, dict):
        events = [
            {
                "type": k,
                **v,
            }
            for k, v in events.items()
            if isinstance(v, dict)
        ]

    events_out = []

    allowed_events = {
        "stable_function",
        "structural_deterioration",
        "prosthetic_stenosis",
        "prosthetic_regurgitation",
        "paravalvular_leak",
        "endocarditis",
        "thrombosis",
        "reintervention",
        "other_prosthetic_dysfunction",
    }

    if isinstance(events, list):

        for e in events:

            if not isinstance(e, dict):
                continue

            event_type = e.get("type")

            if event_type not in allowed_events:
                continue

            ids, text = resolve_evidence_ids(
                e.get("e"),
                evidence_map,
            )

            events_out.append(
                {
                    "event_year": parse_year(
                        e.get("year")
                    ),
                    "event_type": event_type,
                    "severity": e.get("severity"),
                    "led_to_reintervention": (
                        e.get("reintervention")
                    ),
                    "evidence_ids": ids,
                    "evidence": text,
                }
            )

    ambiguities = obj.get(
        "ambiguities",
        [],
    )

    if not isinstance(ambiguities, list):
        ambiguities = []

    return {
        "procedures": procedures_out,
        "prosthetic_events": events_out,
        "ambiguities": ambiguities,
    }


# ------------------------------------------------------------
# 3. DIRECT-EVIDENCE RULES
# ------------------------------------------------------------

YEAR_RE = re.compile(
    r"\b(19\d{2}|20\d{2})\b",
    re.I,
)

TAVR_RE = re.compile(
    r"\b(TAVR|transcatheter\s+aortic\s+valve)\b",
    re.I,
)

SAVR_RE = re.compile(
    r"\b("
    r"SAVR|"
    r"surgical\s+aortic\s+valve|"
    r"bioprosthetic\s+AVR|"
    r"aortic\s+valve\s+replacement|"
    r"AVR"
    r")\b",
    re.I,
)

IMAGING_RE = re.compile(
    r"\b("
    r"echo|echocardiogram|TTE|TEE|"
    r"LVEF|gradient"
    r")\b",
    re.I,
)

COMPLETION_RE = re.compile(
    r"\b("
    r"underwent|"
    r"s/p|"
    r"status\s+post|"
    r"previous|"
    r"prior|"
    r"performed|"
    r"replacement|"
    r"implant(?:ed|ation)?|"
    r"TAVR|SAVR|AVR"
    r")\b",
    re.I,
)

STENOSIS_RE = re.compile(
    r"\b(stenos\w*|AS)\b",
    re.I,
)

REGURG_RE = re.compile(
    r"\b("
    r"regurg\w*|"
    r"insufficien\w*|"
    r"AI|AR"
    r")\b",
    re.I,
)

PVL_RE = re.compile(
    r"\b(paravalv\w*|perivalv\w*)\b",
    re.I,
)

ENDO_RE = re.compile(
    r"\bendocard\w*\b",
    re.I,
)

THROMB_RE = re.compile(
    r"\bthromb\w*\b",
    re.I,
)

SVD_RE = re.compile(
    r"\b("
    r"SVD|"
    r"structural\s+valve\s+deterioration|"
    r"degenerat\w*"
    r")\b",
    re.I,
)

FAILURE_RE = re.compile(
    r"\b(failure|dysfunction)\b",
    re.I,
)

STABLE_RE = re.compile(
    r"\b("
    r"well[-\s]?seated|"
    r"well[-\s]?functioning|"
    r"functioning\s+(normally|well)|"
    r"stable\s+gradient|"
    r"low\s+gradient|"
    r"no\s+(significant\s+)?"
    r"(aortic\s+)?"
    r"(regurgitation|insufficiency)"
    r")\b",
    re.I,
)

LOW_AR_RE = re.compile(
    r"\b(trace|trivial|mild|1\+)\b",
    re.I,
)

MEANINGFUL_AR_RE = re.compile(
    r"\b(moderate|severe|2\+|3\+|4\+)\b",
    re.I,
)

BIOPROSTHETIC_RE = re.compile(
    r"\b("
    r"bioprosth\w*|"
    r"tissue\s+valve|"
    r"pericardial\s+valve"
    r")\b",
    re.I,
)

VIV_RE = re.compile(
    r"\b(valve[-\s]?in[-\s]?valve|ViV)\b",
    re.I,
)

HEADING_RE = re.compile(
    r"^\s*("
    r"impression|assessment|plan|"
    r"history|exam|diagnosis|diagnoses"
    r")\s*:?\s*$",
    re.I,
)


def line_year(text):

    years = [
        int(x)
        for x in YEAR_RE.findall(
            str(text or "")
        )
    ]

    years = sorted(set(years))

    if len(years) == 1:
        return years[0]

    return None


def procedure_supports(
    proc_type,
    text,
):

    if not text:
        return False

    if HEADING_RE.match(text):
        return False

    if proc_type == "TAVR":
        type_ok = bool(
            TAVR_RE.search(text)
        )

    elif proc_type == "SAVR":
        type_ok = bool(
            SAVR_RE.search(text)
        )

    else:
        type_ok = bool(
            TAVR_RE.search(text)
            or
            SAVR_RE.search(text)
        )

    if not type_ok:
        return False

    # An echo that only says which prosthesis exists
    # is not the implantation procedure.
    if IMAGING_RE.search(text):

        explicit_completion = re.search(
            r"\b("
            r"underwent|"
            r"s/p|"
            r"status\s+post|"
            r"previous|"
            r"prior|"
            r"performed|"
            r"replacement|"
            r"implant(?:ed|ation)?"
            r")\b",
            text,
            re.I,
        )

        if not explicit_completion:
            return False

    return bool(
        COMPLETION_RE.search(text)
    )


def event_supports(
    event_type,
    text,
):

    if not text:
        return False

    if HEADING_RE.match(text):
        return False

    if event_type == "prosthetic_stenosis":
        return bool(
            STENOSIS_RE.search(text)
        )

    if event_type == "prosthetic_regurgitation":

        if not REGURG_RE.search(text):
            return False

        # trace/trivial/mild alone is too weak
        if (
            LOW_AR_RE.search(text)
            and not MEANINGFUL_AR_RE.search(text)
        ):
            return False

        return True

    if event_type == "paravalvular_leak":
        return bool(
            PVL_RE.search(text)
        )

    if event_type == "endocarditis":
        return bool(
            ENDO_RE.search(text)
        )

    if event_type == "thrombosis":
        return bool(
            THROMB_RE.search(text)
        )

    if event_type == "structural_deterioration":
        return bool(
            SVD_RE.search(text)
        )

    if event_type == "other_prosthetic_dysfunction":

        if not FAILURE_RE.search(text):
            return False

        # specific mechanism beats generic dysfunction
        if any(
            r.search(text)
            for r in [
                STENOSIS_RE,
                REGURG_RE,
                PVL_RE,
                ENDO_RE,
                THROMB_RE,
                SVD_RE,
            ]
        ):
            return False

        return True

    if event_type == "stable_function":
        return bool(
            STABLE_RE.search(text)
        )

    if event_type == "reintervention":

        return bool(
            (
                TAVR_RE.search(text)
                or SAVR_RE.search(text)
            )
            and re.search(
                r"\b("
                r"underwent|"
                r"s/p|"
                r"status\s+post|"
                r"replacement|"
                r"implant(?:ed|ation)?"
                r")\b",
                text,
                re.I,
            )
        )

    return False


# ------------------------------------------------------------
# 4. REGROUND QWEN FACTS TO THE REAL SELECTED NOTE
# ------------------------------------------------------------

def reground_ids(
    proposed_ids,
    evidence_map,
    validator,
):

    valid = []

    # First try Qwen's IDs
    for i in proposed_ids or []:

        text = evidence_map.get(
            i,
            "",
        )

        if validator(text):
            valid.append(i)

    if valid:
        return valid[:3]

    # If Qwen cited the wrong line, search
    # the selected context deterministically.
    for i, text in evidence_map.items():

        if validator(text):
            valid.append(i)

    return valid[:3]


def support_size(
    size,
    preferred_ids,
    evidence_map,
):

    if size is None:
        return None, []

    try:
        size = int(float(size))
    except Exception:
        return None, []

    patterns = [
        rf"#\s*{size}\b",
        rf"\b{size}\s*mm\b",
        rf"\bsize\s*#?\s*{size}\b",
    ]

    search_ids = list(
        dict.fromkeys(
            list(preferred_ids)
            + list(evidence_map.keys())
        )
    )

    for i in search_ids:

        text = evidence_map.get(
            i,
            "",
        )

        if any(
            re.search(p, text, re.I)
            for p in patterns
        ):
            return size, [i]

    return None, []


def support_model(
    model_name,
    preferred_ids,
    evidence_map,
):

    if not isinstance(
        model_name,
        str,
    ):
        return None, []

    model_name = re.sub(
        r"\s*#?\s*\d{2}\s*(mm)?\s*$",
        "",
        model_name,
    ).strip()

    if (
        not model_name
        or "[NAME]" in model_name
    ):
        return None, []

    target = re.sub(
        r"[^a-z0-9]+",
        " ",
        model_name.lower(),
    ).strip()

    search_ids = list(
        dict.fromkeys(
            list(preferred_ids)
            + list(evidence_map.keys())
        )
    )

    for i in search_ids:

        text = evidence_map.get(
            i,
            "",
        )

        normalized = re.sub(
            r"[^a-z0-9]+",
            " ",
            text.lower(),
        ).strip()

        if target in normalized:
            return model_name, [i]

        # tolerate tiny spelling variants:
        # Biocor vs Biocore
        target_tokens = target.split()
        note_tokens = normalized.split()

        if target_tokens:

            ok = True

            for token in target_tokens:

                if len(token) < 4:
                    continue

                best = max(
                    [
                        difflib.SequenceMatcher(
                            None,
                            token,
                            x,
                        ).ratio()
                        for x in note_tokens
                    ]
                    or [0]
                )

                if best < 0.86:
                    ok = False
                    break

            if ok:
                return model_name, [i]

    return None, []


def evidence_text(
    ids,
    evidence_map,
):

    return " | ".join(
        evidence_map[i]
        for i in ids
        if i in evidence_map
    )


# ------------------------------------------------------------
# 5. FINAL VALIDATOR
#
# Existing qwen_extract_followup() calls this with only obj.
# We use _LAST_EVIDENCE_MAP set by compact_to_canonical().
# ------------------------------------------------------------

def validate_canonical_extraction(obj):

    evidence_map = _LAST_EVIDENCE_MAP

    if not isinstance(obj, dict):
        obj = {}

    # ----------------------------
    # PROCEDURES
    # ----------------------------

    procedures = []

    for p in obj.get(
        "procedures",
        [],
    ):

        if not isinstance(p, dict):
            continue

        proc_type = p.get(
            "procedure_type"
        )

        proc_ids = reground_ids(
            p.get(
                "evidence_ids",
                [],
            ),
            evidence_map,
            lambda text: procedure_supports(
                proc_type,
                text,
            ),
        )

        if not proc_ids:
            continue

        size, size_ids = support_size(
            p.get(
                "valve_size_mm"
            ),
            proc_ids,
            evidence_map,
        )

        model_name, model_ids = support_model(
            p.get(
                "valve_model"
            ),
            proc_ids,
            evidence_map,
        )

        all_ids = list(
            dict.fromkeys(
                proc_ids
                + model_ids
                + size_ids
            )
        )[:3]

        # year ONLY from a line that itself
        # documents that procedure
        years = []

        for i in all_ids:

            text = evidence_map.get(
                i,
                "",
            )

            if procedure_supports(
                proc_type,
                text,
            ):

                y = line_year(text)

                if y is not None:
                    years.append(y)

        years = sorted(
            set(years)
        )

        procedure_year = (
            years[0]
            if len(years) == 1
            else None
        )

        procedures.append(
            {
                "procedure_year": procedure_year,
                "procedure_type": proc_type,
                "valve_model": model_name,
                "valve_size_mm": size,
                "is_redo": p.get(
                    "is_redo"
                ),
                "is_valve_in_valve": p.get(
                    "is_valve_in_valve"
                ),
                "evidence_ids": all_ids,
                "evidence": evidence_text(
                    all_ids,
                    evidence_map,
                ),
            }
        )

    # explicit years first
    procedures.sort(
        key=lambda p: (
            p["procedure_year"] is None,
            p["procedure_year"]
            if p["procedure_year"] is not None
            else 9999,
        )
    )

    all_context = "\n".join(
        evidence_map.values()
    )

    # Repair redo / ViV
    for i, p in enumerate(
        procedures
    ):

        if i == 0:

            p["is_redo"] = False

            if (
                p["procedure_type"]
                != "TAVR"
            ):
                p[
                    "is_valve_in_valve"
                ] = False

            continue

        p["is_redo"] = True

        if (
            p["procedure_type"]
            == "TAVR"
        ):

            explicit_viv = bool(
                VIV_RE.search(
                    all_context
                )
            )

            prior_surgical = any(
                x["procedure_type"]
                in {
                    "SAVR",
                    "AVR_unspecified",
                }
                for x in procedures[:i]
            )

            prior_bioprosthetic = bool(
                BIOPROSTHETIC_RE.search(
                    all_context
                )
            )

            if (
                explicit_viv
                or (
                    prior_surgical
                    and prior_bioprosthetic
                )
            ):
                p[
                    "is_valve_in_valve"
                ] = True

    # ----------------------------
    # EVENTS
    # ----------------------------

    events = []

    for e in obj.get(
        "prosthetic_events",
        [],
    ):

        if not isinstance(e, dict):
            continue

        event_type = e.get(
            "event_type"
        )

        ids = reground_ids(
            e.get(
                "evidence_ids",
                [],
            ),
            evidence_map,
            lambda text: event_supports(
                event_type,
                text,
            ),
        )

        if not ids:
            continue

        ids = ids[:2]

        years = []

        for i in ids:

            text = evidence_map.get(
                i,
                "",
            )

            if event_supports(
                event_type,
                text,
            ):

                y = line_year(text)

                if y is not None:
                    years.append(y)

        years = sorted(
            set(years)
        )

        event_year = (
            years[0]
            if len(years) == 1
            else None
        )

        events.append(
            {
                "event_year": event_year,
                "event_type": event_type,
                "severity": e.get(
                    "severity"
                ),
                "led_to_reintervention": (
                    e.get(
                        "led_to_reintervention"
                    )
                ),
                "evidence_ids": ids,
                "evidence": evidence_text(
                    ids,
                    evidence_map,
                ),
            }
        )

    # Repeat valve replacement = reintervention
    if len(procedures) > 1:

        later = procedures[-1]

        events.append(
            {
                "event_year": (
                    later[
                        "procedure_year"
                    ]
                ),
                "event_type": (
                    "reintervention"
                ),
                "severity": None,
                "led_to_reintervention": True,
                "evidence_ids": (
                    later[
                        "evidence_ids"
                    ]
                ),
                "evidence": (
                    later[
                        "evidence"
                    ]
                ),
            }
        )

    # ----------------------------
    # EVENT DEDUPLICATION
    # ----------------------------

    severity_rank = {
        "": 0,
        "trace": 1,
        "trivial": 1,
        "mild": 2,
        "moderate": 3,
        "severe": 4,
    }

    best = {}

    for e in events:

        t = e[
            "event_type"
        ]

        sev = str(
            e.get(
                "severity"
            )
            or ""
        ).lower()

        score = 0

        for word, rank in (
            severity_rank.items()
        ):

            if (
                word
                and word in sev
            ):
                score = max(
                    score,
                    rank,
                )

        if (
            t not in best
            or score > best[t][0]
        ):
            best[t] = (
                score,
                e,
            )

    events = [
        pair[1]
        for pair in best.values()
    ]

    return {
        "procedures": procedures,
        "prosthetic_events": events,
        "ambiguities": (
            obj.get(
                "ambiguities",
                [],
            )
            if isinstance(
                obj.get(
                    "ambiguities",
                    [],
                ),
                list,
            )
            else []
        ),
    }


print(
    "✓ Final Qwen grounding patch active"
)

✓ Final Qwen grounding patch active


In [20]:
# ============================================================
# FINAL CLOSING PATCH
# 1. collect all supporting procedure lines
# 2. deterministically recover explicit prosthetic events
# ============================================================

# Save the current hybrid validator exactly once.
if "_QWEN_BASE_VALIDATOR" not in globals():
    _QWEN_BASE_VALIDATOR = validate_canonical_extraction


# ------------------------------------------------------------
# Stricter completed-procedure grounding
# ------------------------------------------------------------

COMPLETED_PROCEDURE_RE = re.compile(
    r"\b("
    r"underwent|"
    r"s/p|"
    r"status\s+post|"
    r"previous|"
    r"prior|"
    r"performed|"
    r"replacement|"
    r"implant(?:ed|ation)?|"
    r"bioprosthetic\s+AVR|"
    r"prosthetic\s+AVR"
    r")\b",
    re.I,
)


def procedure_supports(proc_type, text):

    if not text:
        return False

    if HEADING_RE.match(text):
        return False

    if proc_type == "TAVR":
        type_ok = bool(
            TAVR_RE.search(text)
        )

    elif proc_type == "SAVR":
        type_ok = bool(
            SAVR_RE.search(text)
        )

    else:
        type_ok = bool(
            TAVR_RE.search(text)
            or SAVR_RE.search(text)
        )

    if not type_ok:
        return False

    # Merely mentioning a TAVR clinic / existing valve
    # is not evidence of implantation.
    return bool(
        COMPLETED_PROCEDURE_RE.search(text)
    )


def reground_ids(
    proposed_ids,
    evidence_map,
    validator,
):
    """
    Keep valid Qwen evidence AND search for additional
    directly supporting lines.

    This lets one line support the year and another
    support model/size.
    """

    valid = []

    # Qwen evidence first.
    for i in proposed_ids or []:

        text = evidence_map.get(
            i,
            "",
        )

        if (
            validator(text)
            and i not in valid
        ):
            valid.append(i)

    # Then search the complete selected context.
    for i, text in evidence_map.items():

        if (
            validator(text)
            and i not in valid
        ):
            valid.append(i)

    return valid[:3]


# ------------------------------------------------------------
# Deterministic explicit prosthetic-event discovery
# ------------------------------------------------------------

PROSTHETIC_CONTEXT_RE = re.compile(
    r"\b("
    r"prosthe\w*|"
    r"bioprosthe\w*|"
    r"TAVR|"
    r"SAVR|"
    r"AVR|"
    r"transcatheter\s+aortic"
    r")\b",
    re.I,
)


def extract_severity(text):

    text = str(text)

    for label in [
        "severe",
        "moderate",
        "mild",
        "trace",
        "trivial",
    ]:
        if re.search(
            rf"\b{label}\b",
            text,
            re.I,
        ):
            return label

    for label in [
        "4+",
        "3+",
        "2+",
        "1+",
    ]:
        if label in text:
            return label

    return None


def deterministic_event_candidates(
    evidence_map,
):
    """
    Recover events whose terminology is sufficiently explicit
    that an LLM is unnecessary.
    """

    candidates = []

    for evidence_id, text in evidence_map.items():

        if not text:
            continue

        prosthetic_context = bool(
            PROSTHETIC_CONTEXT_RE.search(
                text
            )
        )

        # ------------------------
        # Prosthetic stenosis
        # ------------------------

        if (
            prosthetic_context
            and STENOSIS_RE.search(text)
        ):
            candidates.append(
                {
                    "event_year": line_year(
                        text
                    ),
                    "event_type": (
                        "prosthetic_stenosis"
                    ),
                    "severity": extract_severity(
                        text
                    ),
                    "led_to_reintervention": None,
                    "evidence_ids": [
                        evidence_id
                    ],
                    "evidence": text,
                }
            )

        # ------------------------
        # Paravalvular leak
        # ------------------------

        if PVL_RE.search(text):

            candidates.append(
                {
                    "event_year": line_year(
                        text
                    ),
                    "event_type": (
                        "paravalvular_leak"
                    ),
                    "severity": extract_severity(
                        text
                    ),
                    "led_to_reintervention": None,
                    "evidence_ids": [
                        evidence_id
                    ],
                    "evidence": text,
                }
            )

        # ------------------------
        # Prosthetic regurgitation
        # ------------------------

        if (
            prosthetic_context
            and REGURG_RE.search(text)
        ):

            # Do not treat trace/trivial/mild alone
            # as clinically meaningful regurgitation.
            low_only = bool(
                LOW_AR_RE.search(text)
                and not MEANINGFUL_AR_RE.search(
                    text
                )
            )

            if not low_only:

                candidates.append(
                    {
                        "event_year": line_year(
                            text
                        ),
                        "event_type": (
                            "prosthetic_regurgitation"
                        ),
                        "severity": (
                            extract_severity(
                                text
                            )
                        ),
                        "led_to_reintervention": None,
                        "evidence_ids": [
                            evidence_id
                        ],
                        "evidence": text,
                    }
                )

        # ------------------------
        # Endocarditis
        # ------------------------

        if (
            prosthetic_context
            and ENDO_RE.search(text)
        ):
            candidates.append(
                {
                    "event_year": line_year(
                        text
                    ),
                    "event_type": (
                        "endocarditis"
                    ),
                    "severity": None,
                    "led_to_reintervention": None,
                    "evidence_ids": [
                        evidence_id
                    ],
                    "evidence": text,
                }
            )

        # ------------------------
        # Thrombosis
        # ------------------------

        if (
            prosthetic_context
            and THROMB_RE.search(text)
        ):
            candidates.append(
                {
                    "event_year": line_year(
                        text
                    ),
                    "event_type": (
                        "thrombosis"
                    ),
                    "severity": None,
                    "led_to_reintervention": None,
                    "evidence_ids": [
                        evidence_id
                    ],
                    "evidence": text,
                }
            )

        # ------------------------
        # Structural deterioration
        # ------------------------

        if (
            prosthetic_context
            and SVD_RE.search(text)
        ):
            candidates.append(
                {
                    "event_year": line_year(
                        text
                    ),
                    "event_type": (
                        "structural_deterioration"
                    ),
                    "severity": None,
                    "led_to_reintervention": None,
                    "evidence_ids": [
                        evidence_id
                    ],
                    "evidence": text,
                }
            )

        # ------------------------
        # Generic failure
        # only when no mechanism exists
        # ------------------------

        if (
            prosthetic_context
            and FAILURE_RE.search(text)
            and not any(
                r.search(text)
                for r in [
                    STENOSIS_RE,
                    REGURG_RE,
                    PVL_RE,
                    ENDO_RE,
                    THROMB_RE,
                    SVD_RE,
                ]
            )
        ):
            candidates.append(
                {
                    "event_year": line_year(
                        text
                    ),
                    "event_type": (
                        "other_prosthetic_dysfunction"
                    ),
                    "severity": None,
                    "led_to_reintervention": None,
                    "evidence_ids": [
                        evidence_id
                    ],
                    "evidence": text,
                }
            )

    return candidates


# ------------------------------------------------------------
# Final wrapper
# ------------------------------------------------------------

def validate_canonical_extraction(obj):

    result = _QWEN_BASE_VALIDATOR(
        obj
    )

    evidence_map = _LAST_EVIDENCE_MAP

    deterministic = (
        deterministic_event_candidates(
            evidence_map
        )
    )

    all_events = (
        result[
            "prosthetic_events"
        ]
        + deterministic
    )

    # Prefer strongest evidence per event type.
    severity_score = {
        None: 0,
        "": 0,
        "trace": 1,
        "trivial": 1,
        "1+": 1,
        "mild": 2,
        "2+": 3,
        "moderate": 3,
        "3+": 4,
        "4+": 4,
        "severe": 5,
    }

    best = {}

    for event in all_events:

        event_type = event[
            "event_type"
        ]

        severity = str(
            event.get(
                "severity"
            )
            or ""
        ).lower()

        score = 0

        for key, value in (
            severity_score.items()
        ):

            if (
                key
                and key in severity
            ):
                score = max(
                    score,
                    value,
                )

        # Keep stable/reintervention even
        # though they have no severity.
        if event_type in {
            "stable_function",
            "reintervention",
        }:
            score += 10

        if (
            event_type not in best
            or score
            > best[event_type][0]
        ):
            best[event_type] = (
                score,
                event,
            )

    result[
        "prosthetic_events"
    ] = [
        x[1]
        for x in best.values()
    ]

    return result


print(
    "✓ Final procedure + event grounding patch active"
)

✓ Final procedure + event grounding patch active


In [22]:
# ============================================================
# FINAL PATCH — valve-specific failure/dysfunction only
# ============================================================

# Replace the overly broad:
#   r"\b(failure|dysfunction)\b"
#
# This will still catch:
#   "bioprosthetic failure"
#   "prosthetic valve dysfunction"
#   "aortic valve failure"
#   "prosthesis dysfunction"
#
# But NOT:
#   "diastolic dysfunction"
#   "RV dysfunction"
#   "LV systolic dysfunction"

VALVE_FAILURE_RE = re.compile(
    r"\b(?:"
    r"(?:prosthetic|bioprosthetic)"
        r"(?:\s+aortic)?"
        r"(?:\s+valve|\s+AVR)?"
    r"|"
    r"aortic\s+prosthe\w*"
    r"|"
    r"valve"
    r"|"
    r"prosthesis"
    r")"
    r"\s+(?:failure|dysfunction)\b",
    re.I,
)

# The existing validation/extraction functions refer to FAILURE_RE
# dynamically, so replacing this global is enough.
FAILURE_RE = VALVE_FAILURE_RE


# Quick sanity checks
_tests = {
    "bioprosthetic failure": True,
    "prosthetic valve dysfunction": True,
    "aortic valve failure": True,
    "prosthesis dysfunction": True,
    "grade 2 diastolic dysfunction": False,
    "RV systolic dysfunction": False,
    "LV dysfunction": False,
}

for text, expected in _tests.items():
    observed = bool(
        FAILURE_RE.search(text)
    )

    assert observed == expected, (
        text,
        observed,
        expected,
    )

print(
    "✓ FINAL patch active — "
    "generic cardiac dysfunction can no longer "
    "be mislabeled as prosthetic-valve dysfunction"
)

✓ FINAL patch active — generic cardiac dysfunction can no longer be mislabeled as prosthetic-valve dysfunction


In [23]:
gc.collect()
torch.cuda.empty_cache()

row = benchmark_gold_10.iloc[0]

smoke = extract_valve_note(
    row,
    verbose=True,
)

print(
    "\nMode:",
    smoke["mode"]
)

print(
    "Parse error:",
    smoke["parse_error"]
)

print(
    "Prompt tokens:",
    smoke["prompt_tokens"]
)

print(
    "Generated tokens:",
    smoke["generated_tokens"]
)

print(
    "\nStructured result:"
)

print(
    smoke["result"]
)

print(
    "\nRaw Qwen output:"
)

print(
    smoke["raw_text"]
)

Context: 5339 → 2116 tokens
Actual Qwen prompt: 3412 tokens
Generated tokens: 333 / 1000

Mode: qwen2.5_7b
Parse error: None
Prompt tokens: 3412
Generated tokens: 333

Structured result:
{'procedures': [{'procedure_year': 2010, 'procedure_type': 'SAVR', 'valve_model': 'Biocor', 'valve_size_mm': 27, 'is_redo': False, 'is_valve_in_valve': False, 'evidence_ids': [34, 2, 4], 'evidence': 'Biocore #27 bioprosthetic AVR; with LAA ligation and Maze; [FACILITY], Dr. [NAME] | [NAME] A Parker is a [AGE] male with history of biventricular heart failure with recovered LV function, aortic stenosis, history of bioprosthetic AVR in August 2010, chronic atrial fibrillation atrial flutter, status post maze at the time of his bioprosthetic AVR, history of hypertension, CKD stage III with proteinuria, colon mass with previous resection. | His previous AVR was performed by Dr. [NAME] at [FACILITY] Main campus in August 2010.'}, {'procedure_year': None, 'procedure_type': 'TAVR', 'valve_model': 'Edwards S3',

In [13]:
row = benchmark_gold_10.iloc[0]

smoke = extract_valve_note(
    row,
    verbose=True,
)

print("\nMode:", smoke["mode"])
print("Parse error:", smoke["parse_error"])
print("\nStructured result:")
display(smoke["result"])

print("\nRaw Qwen output:")
print(smoke["raw_text"])

Context: 5339 → 2116 tokens
Actual Qwen prompt: 3382 tokens
Generated tokens: 283 / 1000

Mode: qwen2.5_7b
Parse error: None

Structured result:


{'procedures': [{'procedure_year': 2010,
   'procedure_type': 'SAVR',
   'valve_model': None,
   'valve_size_mm': None,
   'is_redo': False,
   'is_valve_in_valve': False,
   'evidence_ids': [4],
   'evidence': 'His previous AVR was performed by Dr. [NAME] at [FACILITY] Main campus in August 2010.'},
  {'procedure_year': None,
   'procedure_type': 'TAVR',
   'valve_model': None,
   'valve_size_mm': 26,
   'is_redo': True,
   'is_valve_in_valve': False,
   'evidence_ids': [6],
   'evidence': 'He underwent successful TAVR using 26 mm [NAME] valve on [DATE].  Postoperatively his echo revealed normal LV functions, his peak and mean gradients were 20 and 10 mm respectively and dimensionless index 0.37.'}],
 'prosthetic_events': [{'event_year': None,
   'event_type': 'reintervention',
   'severity': None,
   'led_to_reintervention': True,
   'evidence_ids': [6],
   'evidence': 'He underwent successful TAVR using 26 mm [NAME] valve on [DATE].  Postoperatively his echo revealed normal LV funct


Raw Qwen output:
{
  "procedures": [
    {"year":2010,"type":"SAVR","model":"Biocor #27","size":27,"redo":false,"viv":false,"e":4},
    {"year":2018,"type":"TAVR","model":"Edwards S3","size":26,"redo":true,"viv":false,"e":6}
  ],
  "events": [
    {"type":"stable_function","year":null,"severity":null,"reintervention":null,"e":7},
    {"type":"prosthetic_stenosis","year":2018,"severity":"moderate","reintervention":false,"e":27},
    {"type":"prosthetic_regurgitation","year":2018,"severity":"trace to 1+","reintervention":false,"e":27},
    {"type":"paravalvular_leak","year":2018,"severity":"regurgitant jet with anterior perivalvular region origin","reintervention":false,"e":67},
    {"type":"structural_deterioration","year":2018,"severity":"moderate aortic stenosis","reintervention":false,"e":67}
  ],
  "ambiguities": []
}


### Prompt freeze

After the B01 sanity check below, **do not tune this prompt against individual benchmark notes**. If Qwen is structurally stable, run the remaining benchmark unchanged so the comparison is interpretable.


## 12. Frozen 10-note benchmark run

Do not modify the selector or extraction prompt during this run.

In [24]:
model_outputs = []

for i, row in benchmark_gold_10.iterrows():
    bid = row["Benchmark_ID"]
    patient = row["Patient"]

    print("\n" + "=" * 90)
    print(f"{bid} | {patient} | {i+1}/10")
    print("=" * 90)

    gc.collect()
    torch.cuda.empty_cache()

    try:
        out = extract_valve_note(
            row,
            verbose=True,
        )
        error = None

        if out["parse_error"] is None:
            print("✓ extraction complete")
        else:
            print(
                "✗ extraction incomplete:",
                out["parse_error"],
            )

    except Exception as e:
        out = {
            "result": {
                "procedures": [],
                "prosthetic_events": [],
                "ambiguities": [],
            },
            "raw_text": None,
            "selected_context": select_valve_context(
                row["Notes"]
            ),
            "parse_error": None,
            "prompt_tokens": None,
            "mode": "ERROR",
        }
        error = repr(e)
        print("✗ extraction failed:", error)
        traceback.print_exc()

    model_outputs.append(
        {
            "Benchmark_ID": bid,
            "Patient": patient,
            "result": out["result"],
            "raw_text": out["raw_text"],
            "selected_context": out["selected_context"],
            "parse_error": out["parse_error"],
            "prompt_tokens": out["prompt_tokens"],
            "generated_tokens": out.get("generated_tokens"),
            "mode": out["mode"],
            "error": error,
        }
    )

benchmark_predictions = pd.DataFrame(model_outputs)

display(
    benchmark_predictions[
        [
            "Benchmark_ID",
            "Patient",
            "mode",
            "parse_error",
            "error",
        ]
    ]
)


B01 | Patient_035 | 1/10
Context: 5339 → 2116 tokens
Actual Qwen prompt: 3412 tokens
Generated tokens: 333 / 1000
✓ extraction complete

B02 | Patient_042 | 2/10
Context: 4877 → 1235 tokens
Actual Qwen prompt: 2555 tokens
Generated tokens: 221 / 1000
✓ extraction complete

B03 | Patient_086 | 3/10
Context: 3440 → 916 tokens
Actual Qwen prompt: 2119 tokens
Generated tokens: 136 / 1000
✓ extraction complete

B04 | Patient_104 | 4/10
Context: 3072 → 697 tokens
Actual Qwen prompt: 1897 tokens
Generated tokens: 269 / 1000
✓ extraction complete

B05 | Patient_068 | 5/10
Context: 3057 → 737 tokens
Actual Qwen prompt: 1985 tokens
Generated tokens: 600 / 1000
✓ extraction complete

B06 | Patient_044 | 6/10
Context: 1584 → 675 tokens
Actual Qwen prompt: 1852 tokens
Generated tokens: 332 / 1000
✓ extraction complete

B07 | Patient_058 | 7/10
Context: 4422 → 1389 tokens
Actual Qwen prompt: 2735 tokens
Generated tokens: 336 / 1000
✓ extraction complete

B08 | Patient_038 | 8/10
Context: 4086 → 825

,Benchmark_ID,Patient,mode,parse_error,error
0,B01,Patient_035,qwen2.5_7b,None,None
1,B02,Patient_042,qwen2.5_7b,None,None
2,B03,Patient_086,qwen2.5_7b,None,None
3,B04,Patient_104,qwen2.5_7b,None,None
4,B05,Patient_068,qwen2.5_7b,None,None
5,B06,Patient_044,qwen2.5_7b,None,None
6,B07,Patient_058,qwen2.5_7b,None,None
7,B08,Patient_038,qwen2.5_7b,None,None
8,B09,Patient_036,qwen2.5_7b,None,None
9,B10,Patient_103,qwen2.5_7b,None,None


## 13. Flatten predictions and score the benchmark

In [25]:
def norm_year(x):
    if x is None:
        return None

    m = re.search(
        r"\b(19\d{2}|20\d{2})\b",
        str(x),
    )
    return int(m.group(1)) if m else None


def norm_size(x):
    if x is None:
        return None

    m = re.search(
        r"\d+(?:\.\d+)?",
        str(x),
    )
    if not m:
        return None

    v = float(m.group())
    return int(v) if v.is_integer() else v


def prediction_summary(obj):
    if not isinstance(obj, dict):
        obj = {}

    procedures = obj.get("procedures", [])
    events = obj.get("prosthetic_events", [])
    ambiguities = obj.get("ambiguities", [])

    if not isinstance(procedures, list):
        procedures = []
    if not isinstance(events, list):
        events = []

    procedures = [
        x for x in procedures
        if isinstance(x, dict)
    ]
    events = [
        x for x in events
        if isinstance(x, dict)
    ]

    procedure_types = [
        p.get("procedure_type")
        for p in procedures
    ]

    procedure_years = [
        norm_year(p.get("procedure_year"))
        for p in procedures
    ]

    valve_sizes = [
        norm_size(p.get("valve_size_mm"))
        for p in procedures
    ]

    event_types = sorted(
        {
            e.get("event_type")
            for e in events
            if e.get("event_type") is not None
        }
    )

    is_viv = any(
        p.get("is_valve_in_valve") is True
        for p in procedures
    )

    is_redo = any(
        p.get("is_redo") is True
        for p in procedures
    )

    return {
        "pred_procedure_types": procedure_types,
        "pred_procedure_years": procedure_years,
        "pred_valve_sizes": valve_sizes,
        "pred_is_valve_in_valve": is_viv,
        "pred_is_redo": is_redo,
        "pred_event_types": event_types,
        "pred_n_procedures": len(procedures),
        "pred_n_events": len(events),
        "pred_n_ambiguities": (
            len(ambiguities)
            if isinstance(ambiguities, list)
            else 0
        ),
    }


pred_flat = pd.DataFrame(
    benchmark_predictions["result"]
    .apply(prediction_summary)
    .tolist()
)

benchmark_predictions_flat = pd.concat(
    [
        benchmark_predictions.drop(columns=["result"]),
        pred_flat,
    ],
    axis=1,
)

eval_df = benchmark_gold_10.merge(
    benchmark_predictions_flat,
    on=["Benchmark_ID", "Patient"],
    how="left",
)


def as_list(x):
    if isinstance(x, list):
        return x
    if x is None:
        return []
    if isinstance(x, float) and np.isnan(x):
        return []
    return [x]


def exact_list(a, b):
    return as_list(a) == as_list(b)


def exact_set(a, b):
    return set(as_list(a)) == set(as_list(b))


def jaccard(a, b):
    a = set(as_list(a))
    b = set(as_list(b))

    if not a and not b:
        return 1.0

    return len(a & b) / len(a | b)


eval_df["parse_success"] = (
    eval_df["parse_error"].isna()
    & eval_df["error"].isna()
)


def score_if_parsed(row, fn):
    """
    Failed generations are NA, never implicit negative predictions.
    """
    if not row["parse_success"]:
        return np.nan

    return fn(row)


eval_df["procedure_types_exact"] = eval_df.apply(
    lambda r: score_if_parsed(
        r,
        lambda x: exact_list(
            x["gold_procedure_types"],
            x["pred_procedure_types"],
        ),
    ),
    axis=1,
)

eval_df["procedure_years_exact"] = eval_df.apply(
    lambda r: score_if_parsed(
        r,
        lambda x: exact_list(
            x["gold_procedure_years"],
            x["pred_procedure_years"],
        ),
    ),
    axis=1,
)

eval_df["valve_sizes_exact"] = eval_df.apply(
    lambda r: score_if_parsed(
        r,
        lambda x: exact_list(
            x["gold_valve_sizes"],
            x["pred_valve_sizes"],
        ),
    ),
    axis=1,
)

eval_df["viv_correct"] = eval_df.apply(
    lambda r: score_if_parsed(
        r,
        lambda x: (
            x["gold_is_valve_in_valve"]
            == x["pred_is_valve_in_valve"]
        ),
    ),
    axis=1,
)

eval_df["redo_correct"] = eval_df.apply(
    lambda r: score_if_parsed(
        r,
        lambda x: (
            x["gold_is_redo"]
            == x["pred_is_redo"]
        ),
    ),
    axis=1,
)

eval_df["event_types_exact"] = eval_df.apply(
    lambda r: score_if_parsed(
        r,
        lambda x: exact_set(
            x["gold_event_types"],
            x["pred_event_types"],
        ),
    ),
    axis=1,
)

eval_df["event_jaccard"] = eval_df.apply(
    lambda r: score_if_parsed(
        r,
        lambda x: jaccard(
            x["gold_event_types"],
            x["pred_event_types"],
        ),
    ),
    axis=1,
)


# Conditional extraction accuracy among successfully parsed notes.
summary = pd.Series(
    {
        "n_notes": len(eval_df),
        "n_parse_success": int(
            eval_df["parse_success"].sum()
        ),
        "parse_success_rate": (
            eval_df["parse_success"].mean()
        ),
        "procedure_type_exact_accuracy_among_parsed": (
            eval_df.loc[
                eval_df["parse_success"],
                "procedure_types_exact",
            ].mean()
        ),
        "procedure_year_exact_accuracy_among_parsed": (
            eval_df.loc[
                eval_df["parse_success"],
                "procedure_years_exact",
            ].mean()
        ),
        "valve_size_exact_accuracy_among_parsed": (
            eval_df.loc[
                eval_df["parse_success"],
                "valve_sizes_exact",
            ].mean()
        ),
        "ViV_accuracy_among_parsed": (
            eval_df.loc[
                eval_df["parse_success"],
                "viv_correct",
            ].mean()
        ),
        "redo_accuracy_among_parsed": (
            eval_df.loc[
                eval_df["parse_success"],
                "redo_correct",
            ].mean()
        ),
        "event_set_exact_accuracy_among_parsed": (
            eval_df.loc[
                eval_df["parse_success"],
                "event_types_exact",
            ].mean()
        ),
        "mean_event_jaccard_among_parsed": (
            eval_df.loc[
                eval_df["parse_success"],
                "event_jaccard",
            ].mean()
        ),
    }
)

display(
    summary.to_frame("score")
)

display(
    eval_df[
        [
            "Benchmark_ID",
            "Patient",
            "parse_success",
            "procedure_types_exact",
            "procedure_years_exact",
            "valve_sizes_exact",
            "viv_correct",
            "redo_correct",
            "event_types_exact",
            "event_jaccard",
        ]
    ]
)


,score
n_notes,10.000000
n_parse_success,10.000000
parse_success_rate,1.000000
procedure_type_exact_accuracy_among_parsed,0.600000
procedure_year_exact_accuracy_among_parsed,0.300000
valve_size_exact_accuracy_among_parsed,0.500000
ViV_accuracy_among_parsed,0.900000
redo_accuracy_among_parsed,0.800000
event_set_exact_accuracy_among_parsed,0.200000
mean_event_jaccard_among_parsed,0.521667


,Benchmark_ID,Patient,parse_success,procedure_types_exact,procedure_years_exact,valve_sizes_exact,viv_correct,redo_correct,event_types_exact,event_jaccard
0,B01,Patient_035,True,True,True,True,True,True,True,1.000000
1,B02,Patient_042,True,True,True,True,True,True,False,0.666667
2,B03,Patient_086,True,True,False,True,True,True,False,0.000000
3,B04,Patient_104,True,False,False,False,True,False,False,0.400000
4,B05,Patient_068,True,True,True,True,False,True,False,0.500000
5,B06,Patient_044,True,False,False,False,True,True,False,0.666667
6,B07,Patient_058,True,False,False,False,True,True,True,1.000000
7,B08,Patient_038,True,False,False,False,True,False,False,0.333333
8,B09,Patient_036,True,True,False,True,True,True,False,0.250000
9,B10,Patient_103,True,True,False,False,True,True,False,0.400000


## 14. Error inspection

Use these helpers **after** the frozen benchmark finishes. They show the gold labels, selected context, parsed prediction, and raw model output for one benchmark case.

In [26]:
def inspect_benchmark_result(benchmark_id):
    gold_row = eval_df[
        eval_df["Benchmark_ID"].eq(benchmark_id)
    ].iloc[0]

    pred_row = benchmark_predictions[
        benchmark_predictions["Benchmark_ID"].eq(
            benchmark_id
        )
    ].iloc[0]

    print("=" * 90)
    print(
        benchmark_id,
        "|",
        gold_row["Patient"],
    )
    print("=" * 90)

    print("\nGOLD PROCEDURES:")
    print(gold_row["gold_procedure_types"])
    print(gold_row["gold_procedure_years"])
    print(gold_row["gold_valve_sizes"])

    print("\nGOLD EVENTS:")
    print(gold_row["gold_event_types"])

    print("\nSELECTED CONTEXT:")
    print(pred_row["selected_context"])

    print("\nPARSED PREDICTION:")
    display(
        prediction_summary(
            pred_row["result"]
            if "result" in pred_row
            else {}
        )
    )

    print("\nRAW MODEL OUTPUT:")
    print(pred_row["raw_text"])

    print("\nPARSE ERROR:")
    print(pred_row["parse_error"])


# Example:
# inspect_benchmark_result("B01")

In [29]:
# ============================================================
# MULTIMODAL COHORT — notes + labs + medications
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import gc
import torch

LABS_FILE = Path("labs_deidentified.xlsx")
MEDS_FILE = Path("medication_dataset_EDA.xlsx")

# Colab fallback
if not LABS_FILE.exists():
    LABS_FILE = Path("/mnt/data/labs_deidentified.xlsx")

if not MEDS_FILE.exists():
    MEDS_FILE = Path("/mnt/data/medication_dataset_EDA.xlsx")


labs = pd.read_excel(
    LABS_FILE
)

meds = pd.read_excel(
    MEDS_FILE,
    sheet_name="Dedup View",
)


note_patients = set(
    notes["Patient"]
    .dropna()
    .astype(str)
)

lab_patients = set(
    labs["Patient"]
    .dropna()
    .astype(str)
)

med_patients = set(
    meds["Patient"]
    .dropna()
    .astype(str)
)


multimodal_patients = sorted(
    note_patients
    & lab_patients
    & med_patients
)

print(
    "Multimodal patients:",
    len(multimodal_patients)
)

print(multimodal_patients)


assert len(multimodal_patients) == 17, (
    "Expected 17 multimodal patients, "
    f"found {len(multimodal_patients)}"
)


cohort_notes = (
    notes[
        notes["Patient"].isin(
            multimodal_patients
        )
    ]
    .copy()
    .sort_values(
        ["Patient", "Service Date"]
    )
    .reset_index(drop=True)
)


# Important sanity check:
# there should be exactly one note per patient.
note_counts = (
    cohort_notes
    .groupby("Patient")
    .size()
)

display(
    note_counts.to_frame(
        "n_notes"
    )
)

assert (
    note_counts.eq(1).all()
), "Some multimodal patients have >1 note."


print(
    "\n✓ 17 patients"
    "\n✓ notes present"
    "\n✓ labs present"
    "\n✓ medications present"
    "\n✓ exactly one note per patient"
)

Multimodal patients: 17
['Patient_101', 'Patient_102', 'Patient_103', 'Patient_104', 'Patient_105', 'Patient_106', 'Patient_107', 'Patient_108', 'Patient_109', 'Patient_110', 'Patient_111', 'Patient_112', 'Patient_113', 'Patient_114', 'Patient_115', 'Patient_116', 'Patient_117']


,n_notes
Patient,
Patient_101,1
Patient_102,1
Patient_103,1
Patient_104,1
Patient_105,1
Patient_106,1
Patient_107,1
Patient_108,1
Patient_109,1



✓ 17 patients
✓ notes present
✓ labs present
✓ medications present
✓ exactly one note per patient


In [30]:
# ============================================================
# COHORT COVERAGE SUMMARY
# ============================================================

coverage = (
    pd.DataFrame(
        {
            "Patient":
                multimodal_patients
        }
    )
)


lab_counts = (
    labs
    .groupby("Patient")
    .size()
    .rename("n_labs")
)


med_counts = (
    meds
    .groupby("Patient")
    .size()
    .rename("n_medications")
)


note_counts = (
    cohort_notes
    .groupby("Patient")
    .size()
    .rename("n_notes")
)


coverage = (
    coverage
    .merge(
        note_counts,
        on="Patient",
        how="left",
    )
    .merge(
        lab_counts,
        on="Patient",
        how="left",
    )
    .merge(
        med_counts,
        on="Patient",
        how="left",
    )
)


display(coverage)

print("\nTotals")
print(
    "Patients:",
    len(coverage)
)

print(
    "Lab rows:",
    int(
        coverage["n_labs"].sum()
    )
)

print(
    "Medication rows:",
    int(
        coverage["n_medications"].sum()
    )
)

,Patient,n_notes,n_labs,n_medications
0,Patient_101,1,1396,125
1,Patient_102,1,1825,147
2,Patient_103,1,3164,329
3,Patient_104,1,1640,71
4,Patient_105,1,3130,86
5,Patient_106,1,4084,133
6,Patient_107,1,1012,89
7,Patient_108,1,1775,61
8,Patient_109,1,1661,50
9,Patient_110,1,1138,63



Totals
Patients: 17
Lab rows: 43550
Medication rows: 2157


In [31]:
# ============================================================
# QWEN EXTRACTION — 17 MULTIMODAL PATIENTS
# ============================================================

cohort_outputs = []


for i, row in cohort_notes.iterrows():

    patient = row["Patient"]

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"{patient} | "
        f"{i + 1}/{len(cohort_notes)}"
    )

    print(
        "=" * 90
    )


    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    try:

        out = extract_valve_note(
            row,
            verbose=False,
        )

        error = None

        print(
            "mode:",
            out["mode"],
            "| parse:",
            out["parse_error"],
            "| generated:",
            out.get(
                "generated_tokens"
            ),
        )


    except Exception as e:

        error = repr(e)

        print(
            "✗ ERROR:",
            error,
        )

        out = {
            "result": {
                "procedures": [],
                "prosthetic_events": [],
                "ambiguities": [],
            },
            "raw_text": None,
            "selected_context":
                select_valve_context(
                    row["Notes"]
                ),
            "parse_error":
                "runtime_error",
            "prompt_tokens":
                None,
            "generated_tokens":
                None,
            "mode":
                "ERROR",
        }


    cohort_outputs.append(
        {
            "Patient":
                patient,

            "Note_Row_ID":
                row["Note_Row_ID"],

            "Service_Date":
                row["Service Date"],

            "Note_Type":
                row["Type"],

            "result":
                out["result"],

            "selected_context":
                out["selected_context"],

            "raw_text":
                out["raw_text"],

            "parse_error":
                out["parse_error"],

            "error":
                error,

            "prompt_tokens":
                out["prompt_tokens"],

            "generated_tokens":
                out.get(
                    "generated_tokens"
                ),

            "mode":
                out["mode"],
        }
    )


cohort_predictions = pd.DataFrame(
    cohort_outputs
)


print("\n" + "=" * 90)

print(
    "Parse success:",
    (
        cohort_predictions[
            "parse_error"
        ].isna()
        &
        cohort_predictions[
            "error"
        ].isna()
    ).sum(),
    "/",
    len(cohort_predictions),
)


display(
    cohort_predictions[
        [
            "Patient",
            "Service_Date",
            "mode",
            "parse_error",
            "error",
            "prompt_tokens",
            "generated_tokens",
        ]
    ]
)


Patient_101 | 1/17
mode: qwen2.5_7b | parse: None | generated: 142

Patient_102 | 2/17
mode: qwen2.5_7b | parse: None | generated: 173

Patient_103 | 3/17
mode: qwen2.5_7b | parse: None | generated: 179

Patient_104 | 4/17
mode: qwen2.5_7b | parse: None | generated: 269

Patient_105 | 5/17
mode: qwen2.5_7b | parse: None | generated: 126

Patient_106 | 6/17
mode: qwen2.5_7b | parse: None | generated: 145

Patient_107 | 7/17
mode: qwen2.5_7b | parse: None | generated: 124

Patient_108 | 8/17
mode: qwen2.5_7b | parse: None | generated: 135

Patient_109 | 9/17
mode: qwen2.5_7b | parse: None | generated: 133

Patient_110 | 10/17
mode: qwen2.5_7b | parse: None | generated: 140

Patient_111 | 11/17
mode: qwen2.5_7b | parse: None | generated: 130

Patient_112 | 12/17
mode: qwen2.5_7b | parse: None | generated: 197

Patient_113 | 13/17
mode: qwen2.5_7b | parse: None | generated: 139

Patient_114 | 14/17
mode: qwen2.5_7b | parse: None | generated: 125

Patient_115 | 15/17
mode: qwen2.5_7b | par

,Patient,Service_Date,mode,parse_error,error,prompt_tokens,generated_tokens
0,Patient_101,2022,qwen2.5_7b,None,None,2064,142
1,Patient_102,2019,qwen2.5_7b,None,None,1607,173
2,Patient_103,2023,qwen2.5_7b,None,None,1546,179
3,Patient_104,2022,qwen2.5_7b,None,None,1897,269
4,Patient_105,2026,qwen2.5_7b,None,None,2029,126
5,Patient_106,2024,qwen2.5_7b,None,None,1672,145
6,Patient_107,2012,qwen2.5_7b,None,None,1769,124
7,Patient_108,2014,qwen2.5_7b,None,None,1517,135
8,Patient_109,2011,qwen2.5_7b,None,None,1375,133
9,Patient_110,2011,qwen2.5_7b,None,None,1963,140


In [32]:
# ============================================================
# PER-PATIENT VALVE DATASET — 17 PATIENTS
# ============================================================

import pandas as pd
import numpy as np


def build_patient_valve_row(row):

    patient = row["Patient"]
    result = row["result"]

    if not isinstance(result, dict):
        result = {}

    procedures = result.get(
        "procedures",
        [],
    )

    events = result.get(
        "prosthetic_events",
        [],
    )

    # --------------------------------------------------------
    # Procedures
    # --------------------------------------------------------

    procedure_types = [
        p.get("procedure_type")
        for p in procedures
    ]

    procedure_years = [
        p.get("procedure_year")
        for p in procedures
    ]

    valve_models = [
        p.get("valve_model")
        for p in procedures
    ]

    valve_sizes = [
        p.get("valve_size_mm")
        for p in procedures
    ]

    redo_flags = [
        p.get("is_redo")
        for p in procedures
    ]

    viv_flags = [
        p.get("is_valve_in_valve")
        for p in procedures
    ]

    procedure_evidence = [
        p.get("evidence")
        for p in procedures
    ]

    # First and most recent detected valve procedure
    first_proc = (
        procedures[0]
        if procedures
        else {}
    )

    latest_proc = (
        procedures[-1]
        if procedures
        else {}
    )

    # --------------------------------------------------------
    # Events
    # --------------------------------------------------------

    event_types = [
        e.get("event_type")
        for e in events
    ]

    event_years = [
        e.get("event_year")
        for e in events
    ]

    event_severities = [
        e.get("severity")
        for e in events
    ]

    event_evidence = [
        e.get("evidence")
        for e in events
    ]

    event_set = set(
        x
        for x in event_types
        if x is not None
    )

    # --------------------------------------------------------
    # One patient row
    # --------------------------------------------------------

    return {
        "Patient":
            patient,

        "Service_Date":
            row["Service_Date"],

        # ----------------------
        # Overall valve history
        # ----------------------

        "n_valve_procedures":
            len(procedures),

        "procedure_types":
            procedure_types,

        "procedure_years":
            procedure_years,

        "valve_models":
            valve_models,

        "valve_sizes_mm":
            valve_sizes,

        "redo_flags":
            redo_flags,

        "ViV_flags":
            viv_flags,

        "has_redo":
            any(x is True for x in redo_flags),

        "has_ViV":
            any(x is True for x in viv_flags),

        # ----------------------
        # First valve procedure
        # ----------------------

        "first_procedure_type":
            first_proc.get(
                "procedure_type"
            ),

        "first_procedure_year":
            first_proc.get(
                "procedure_year"
            ),

        "first_valve_model":
            first_proc.get(
                "valve_model"
            ),

        "first_valve_size_mm":
            first_proc.get(
                "valve_size_mm"
            ),

        # ----------------------
        # Latest valve procedure
        # ----------------------

        "latest_procedure_type":
            latest_proc.get(
                "procedure_type"
            ),

        "latest_procedure_year":
            latest_proc.get(
                "procedure_year"
            ),

        "latest_valve_model":
            latest_proc.get(
                "valve_model"
            ),

        "latest_valve_size_mm":
            latest_proc.get(
                "valve_size_mm"
            ),

        # ----------------------
        # Valve events
        # ----------------------

        "n_valve_events":
            len(events),

        "event_types":
            event_types,

        "event_years":
            event_years,

        "event_severities":
            event_severities,

        "has_prosthetic_stenosis":
            "prosthetic_stenosis"
            in event_set,

        "has_prosthetic_regurgitation":
            "prosthetic_regurgitation"
            in event_set,

        "has_paravalvular_leak":
            "paravalvular_leak"
            in event_set,

        "has_structural_deterioration":
            "structural_deterioration"
            in event_set,

        "has_endocarditis":
            "endocarditis"
            in event_set,

        "has_thrombosis":
            "thrombosis"
            in event_set,

        "has_other_prosthetic_dysfunction":
            "other_prosthetic_dysfunction"
            in event_set,

        "has_reintervention":
            "reintervention"
            in event_set,

        "has_stable_function":
            "stable_function"
            in event_set,

        # ----------------------
        # Traceability
        # ----------------------

        "procedure_evidence":
            procedure_evidence,

        "event_evidence":
            event_evidence,
    }


valve_patient_data = pd.DataFrame(
    [
        build_patient_valve_row(row)
        for _, row
        in cohort_predictions.iterrows()
    ]
)


valve_patient_data = (
    valve_patient_data
    .sort_values("Patient")
    .reset_index(drop=True)
)


print(
    "Patients:",
    len(valve_patient_data)
)

display(
    valve_patient_data
)

Patients: 17


,Patient,Service_Date,n_valve_procedures,procedure_types,procedure_years,valve_models,valve_sizes_mm,redo_flags,ViV_flags,has_redo,...,has_prosthetic_regurgitation,has_paravalvular_leak,has_structural_deterioration,has_endocarditis,has_thrombosis,has_other_prosthetic_dysfunction,has_reintervention,has_stable_function,procedure_evidence,event_evidence
0,Patient_101,2022,1,[TAVR],[None],[Edwards-Sapien],[29],[False],[False],False,...,True,False,False,False,False,False,False,False,[He underwent successful TF-TAVR with a 29 mm ...,"[1. Severe AS s/p TF-TAVR, - S/P transcatheter..."
1,Patient_102,2019,1,[SAVR],[None],[trifecta],[23],[False],[False],False,...,True,False,False,False,False,False,False,False,[She is a patient of Dr. [NAME]'s. She has a h...,"[regurgitation. The peak gradient is 16 mmHg, ..."
2,Patient_103,2023,2,"[TAVR, SAVR]","[2019, 2019]","[None, None]","[23, None]","[False, True]","[False, False]",True,...,True,False,False,True,False,False,True,False,[is a [AGE] male patient of Dr. [NAME] with di...,[is a [AGE] male patient of Dr. [NAME] with di...
3,Patient_104,2022,2,"[SAVR, TAVR]","[None, None]",[Carpentier-Edwards prosthetic aortic valve (s...,"[25, None]","[False, True]","[False, False]",True,...,True,True,False,False,False,True,True,False,[Mr. [NAME] is an [AGE] male with AS s/p AVR p...,[severe (3+) aortic valve regurgitation due to...
4,Patient_105,2026,1,[TAVR],[None],[NAME],[26],[False],[False],False,...,False,False,False,False,False,False,False,False,[- The right ventricle is normal in size. Righ...,[Acute systolic (congestive) heart failure (HC...
5,Patient_106,2024,1,[SAVR],[2015],[Trifecta],[23],[False],[False],False,...,True,False,False,False,False,False,False,False,[- Exam indication: HOCM; S/P Myectomy; AVR | ...,[He has a past medical history of Hypertrophic...
6,Patient_107,2012,1,[SAVR],[None],[Perimount],[23],[False],[False],False,...,False,False,False,False,False,False,False,False,"[OPERATIONS: Median sternotomy, coronary arter...",[AS: S/P AVR.]
7,Patient_108,2014,1,[SAVR],[None],[Carpentier-Edwards pericardial],[25],[False],[False],False,...,True,False,False,False,False,False,False,False,[2. Aortic valve replacement utilizing 25-mm C...,[pulmonary hypertension. - AVR (#25 CE). No A...
8,Patient_109,2011,1,[SAVR],[None],[Carpentier-Edwards bovine pericardial],[23],[False],[False],False,...,True,False,False,False,False,False,False,True,"[Critical stenosis, AVA 0.6. Normal EF: 65%. s...","[Critical stenosis, AVA 0.6. Normal EF: 65%. s..."
9,Patient_110,2011,1,[SAVR],[None],[23-mm pericardial prosthesis],[23],[False],[False],False,...,True,False,False,False,False,False,False,True,[• Aortic Valve Replacement [DATE] | PMHx: Sym...,[PMHx: Symptomatic AS with pre-op echo showing...


In [33]:
# ============================================================
# SAVE PER-PATIENT VALVE DATA
# ============================================================

valve_patient_data.to_csv(
    "valve_patient_data_17.csv",
    index=False,
)

valve_patient_data.to_pickle(
    "valve_patient_data_17.pkl"
)

print(
    "✓ Saved valve_patient_data_17.csv"
)
print(
    "✓ Saved valve_patient_data_17.pkl"
)

✓ Saved valve_patient_data_17.csv
✓ Saved valve_patient_data_17.pkl
